# Bayern Munich signature-scene selection from synced 3D football skeleton data

This notebook turns the available Bundesliga 3D skeleton data into per-player MP4 signature-scene animations for **Bayern Munich players**: repeated enough to be recognisable, distinctive versus other Bayern players, and salient enough to be useful for coaching, scouting, or fan-facing storytelling.

The repo data contains five matches, but this notebook deliberately filters to matches involving Bayern Munich and then filters candidate scenes to Bayern players only. That keeps the analysis faster, reduces memory pressure, and makes the notion of a `signature` more coherent for the current dataset.

Each match has:

- `*.parquet`: frame-level 3D skeleton and ball data. Each row is a frame with `frame_number`, `skeletons`, `ball`, `skeleton_count`, and feed metadata. Skeleton targets contain `team`, `jersey_number`, and 21 body parts. The parquet metadata also carries the absolute start/end frames for each in-play phase; frames outside those phase ranges are ignored.
- `kpi_data_*.xml`: synced match events with `SyncedFrameId` anchors. These are used for event candidates because `Events_*.xml` is not aligned to the 3D skeleton frames.
- `Positions_*.xml`: synced 2D tracking frame counters used to map KPI `SyncedFrameId` values into absolute parquet `frame_number` values.
- `MatchInformations_*.xml`: team/player identities, shirts, formations, match metadata, and stadium context.
- `*_metadata.json`: phase frame ranges, FPS, pitch size, team IDs, and player active intervals.

The skeleton docs define the 21 Tracab parts: ears, nose, shoulders, neck, elbows, wrists, hips, pelvis, knees, ankles, heels, and toes. The parquet metadata says the data is captured at 25/30/50/60 Hz; these files are 50 Hz. Team codes inside skeleton targets are `1=home`, `0=away`, `3=referee`.


## Idea

A signature scene is not just an anomaly. It should satisfy four conditions:

1. **Player-specific**: it is closer to that player's other movements than to other players' movements.
2. **Repeatable**: there are similar snippets from the same player elsewhere.
3. **Motion-rich**: the body is doing something expressive enough to be visible in a clip.
4. **Contextual**: the scene is near a meaningful synced KPI football action when possible, such as a shot, tackle, carry, reception, or pass.

The notebook implements this as a weakly supervised retrieval problem:

- Build candidate windows from synced KPI event anchors and optional in-play background windows.
- Extract pose-normalized 3D skeleton sequences for the involved player.
- Convert each sequence into a movement embedding with normalized joint trajectories, joint velocities, and summary biomechanics.
- Score each scene with a combination of distinctiveness, repeatability, and motion energy.
- Render the top three signature scenes for each Bayern player and top-level event into one folder per player, with event-aware filenames and a concise player/event header at the top of each animation.


## Would a foundation model help?

Yes, but with an important caveat: five matches are enough for a prototype, not enough to train a true foundation model from scratch.

A useful movement foundation model would be a self-supervised encoder for football body motion. It would ingest skeleton sequences, ball trajectories, player/team context, and maybe event text. Pretraining tasks could include masked joint reconstruction, future pose prediction, contrastive learning between augmented views of the same movement, and event-aligned prediction. Once trained on many seasons, the encoder would make this notebook stronger because `signature` becomes a retrieval/ranking task in a learned movement space rather than a hand-crafted feature space.

For this hackathon-sized dataset, the pragmatic version is: build the full data pipeline now, use transparent features and nearest-neighbor ranking, then later replace the `scene_features(...)` function with `foundation_model.encode(scene)` without changing candidate generation, scoring, exports, or visualization.


In [1]:
from pathlib import Path
import json
import math
import random
import warnings
import xml.etree.ElementTree as ET
from collections import Counter

import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import pyarrow.parquet as pq
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

try:
    import plotly.express as px
    import plotly.graph_objects as go
except Exception:
    px = None
    go = None

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)


In [2]:
# -----------------------------
# Configuration
# -----------------------------
DATA_DIR = Path("data")
OUTPUT_DIR = Path("signature_scene_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_SEED = 7
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# Focus: only Bayern matches, only Bayern players.
TARGET_TEAM_DISPLAY = "FC Bayern München"
TARGET_TEAM_NUMERIC_ID = 10
TARGET_TEAM_DFL_ID = "DFL-CLU-00000G"
TARGET_TEAM_ALIASES = {"fc bayern münchen", "fc bayern munchen", "bayern", "fcb"}
TARGET_TEAM_ONLY = True
REQUIRE_TARGET_TEAM_MATCHES = True

# Event selection. Set to None or an empty set to use all synced KPI event types.
SELECTED_EVENT_TYPES = {"Carry"}

# Window shape. A 9 second clip gives enough preparation/action/recovery context.
WINDOW_BEFORE_SECONDS = 4.0
WINDOW_AFTER_SECONDS = 5.0

# Skeleton extraction stride. At 50 Hz, stride 8 gives about 6.25 samples/sec.
FRAME_STRIDE = 8
RESAMPLE_STEPS = 24
MIN_FRAMES_FOUND = 10

# Runtime controls. QUICK_MODE keeps the first run responsive; set QUICK_MODE=False for the full candidate set.
QUICK_MODE = False
MATCH_LIMIT = None              # None -> all Bayern matches
MAX_CANDIDATES = 96 if QUICK_MODE else None
MAX_SCENES_PER_PLAYER = 36 if QUICK_MODE else None
BACKGROUND_WINDOWS_PER_PLAYER = 0 if SELECTED_EVENT_TYPES else (2 if QUICK_MODE else 8)
DEDUPLICATE_CANDIDATES = QUICK_MODE  # full mode keeps every selected-event candidate.
MIN_CANDIDATE_GAP_FRAMES = 50   # used only when DEDUPLICATE_CANDIDATES=True
MIN_BATCH_CANDIDATES_PER_ROW_GROUP = 3  # only used by the optional row-group batch extractor

# Full parquet files have many row groups, so small frame filters and row-group batch reads are faster there.
PREFER_ROW_GROUPED_PARQUET_FOR_WINDOWS = True

PART_NAMES = {
    1: "l_ear", 2: "nose", 3: "r_ear", 4: "l_shoulder", 5: "neck", 6: "r_shoulder",
    7: "l_elbow", 8: "r_elbow", 9: "l_wrist", 10: "r_wrist", 11: "l_hip", 12: "pelvis",
    13: "r_hip", 14: "l_knee", 15: "r_knee", 16: "l_ankle", 17: "r_ankle",
    18: "l_heel", 19: "l_toe", 20: "r_heel", 21: "r_toe",
}

BONES = [
    (12, 5), (5, 2), (2, 1), (2, 3),
    (5, 4), (4, 7), (7, 9),
    (5, 6), (6, 8), (8, 10),
    (12, 11), (11, 14), (14, 16), (16, 18), (18, 19),
    (12, 13), (13, 15), (15, 17), (17, 20), (20, 21),
    (11, 13), (4, 6),
]



## Discover Bayern match assets

The loader pairs each Bayern-involved match folder with its parquet skeleton data, synced KPI XML, positions XML, match-info XML, optional unsynced event XML, and metadata JSON. For scene extraction we prefer row-grouped full parquet files because we can read only row groups that overlap selected in-play windows instead of loading a whole match into memory.


In [3]:
def find_single(match_dir: Path, pattern: str):
    found = sorted(match_dir.glob(pattern))
    return found[0] if found else None


def parquet_metadata(path: Path) -> dict:
    md = pq.ParquetFile(path).metadata.metadata or {}
    out = {}
    for key, value in md.items():
        key_text = key.decode(errors="ignore")
        if key_text == "ARROW:schema":
            continue
        out[key_text] = value.decode(errors="ignore")
    return out


def normalise_team_text(value) -> str:
    return str(value or "").casefold().replace("ü", "u")


def metadata_team_is_target(team: dict) -> bool:
    if int(team.get("TeamID", -1)) == TARGET_TEAM_NUMERIC_ID:
        return True
    values = [team.get("LongName"), team.get("ShortName"), team.get("ThreeLetterCode")]
    values = {normalise_team_text(value) for value in values if value}
    aliases = {normalise_team_text(value) for value in TARGET_TEAM_ALIASES}
    return bool(values & aliases)


def select_skeleton_parquet(match_dir: Path) -> Path | None:
    full = [p for p in sorted(match_dir.glob("*.parquet")) if not p.name.endswith("_downsampled.parquet")]
    downsampled = sorted(match_dir.glob("*_downsampled.parquet"))
    if PREFER_ROW_GROUPED_PARQUET_FOR_WINDOWS and full:
        return full[0]
    if downsampled:
        return downsampled[0]
    return full[0] if full else None


def discover_matches(data_dir: Path = DATA_DIR) -> pd.DataFrame:
    rows = []
    for match_dir in sorted(data_dir.iterdir()):
        if not match_dir.is_dir():
            continue
        parquet_path = select_skeleton_parquet(match_dir)
        metadata_path = find_single(match_dir, "*_metadata.json")
        kpi_path = find_single(match_dir, "kpi_data_*.xml")
        positions_path = find_single(match_dir, "Positions_*.xml")
        matchinfo_path = find_single(match_dir, "MatchInformations_*.xml")
        events_path = find_single(match_dir, "Events_*.xml")
        if not (parquet_path and metadata_path and kpi_path and positions_path and matchinfo_path):
            continue

        meta = json.loads(Path(metadata_path).read_text())
        home_is_target = metadata_team_is_target(meta.get("HomeTeam", {}))
        away_is_target = metadata_team_is_target(meta.get("AwayTeam", {}))
        if REQUIRE_TARGET_TEAM_MATCHES and not (home_is_target or away_is_target):
            continue

        target_role = "home" if home_is_target else "guest" if away_is_target else None
        target_team_code = 1 if target_role == "home" else 0 if target_role == "guest" else None
        target_team_name = meta.get("HomeTeam" if home_is_target else "AwayTeam", {}).get("LongName")

        pf = pq.ParquetFile(parquet_path)
        rows.append({
            "match_key": match_dir.name,
            "match_dir": match_dir,
            "parquet_path": parquet_path,
            "kpi_path": kpi_path,
            "positions_path": positions_path,
            "events_path": events_path,
            "matchinfo_path": matchinfo_path,
            "metadata_path": metadata_path,
            "target_role": target_role,
            "target_team_code": target_team_code,
            "target_team_name": target_team_name,
            "parquet_rows": pf.metadata.num_rows,
            "row_groups": pf.metadata.num_row_groups,
        })
    df = pd.DataFrame(rows)
    if MATCH_LIMIT:
        df = df.head(MATCH_LIMIT)
    return df


matches = discover_matches()
matches[["match_key", "target_team_name", "target_role", "target_team_code", "parquet_path", "kpi_path", "parquet_rows", "row_groups"]]


,match_key,target_team_name,target_role,target_team_code,parquet_path,kpi_path,parquet_rows,row_groups
0,2025_09_13_FCB_HSV,FC Bayern München,home,1,data/2025_09_13_FCB_HSV/FCB-HSV.parquet,data/2025_09_13_FCB_HSV/kpi_data_Bayern_Hambur...,419175,280
1,2025_10_04_SGE_FCB,FC Bayern München,guest,0,data/2025_10_04_SGE_FCB/SGE-FCB.parquet,data/2025_10_04_SGE_FCB/kpi_data_Frankfurt_Bay...,383525,256
2,2025_11_08_FCU_FCB,FC Bayern München,guest,0,data/2025_11_08_FCU_FCB/FCU-FCB.parquet,data/2025_11_08_FCU_FCB/kpi_data_Union_Bayern.xml,392276,262


In [4]:
def summarize_match_files(matches: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for asset in matches.to_dict("records"):
        meta = json.loads(Path(asset["metadata_path"]).read_text())
        md = parquet_metadata(asset["parquet_path"])
        root = ET.parse(asset["kpi_path"]).getroot()
        event_types = Counter()
        synced_types = Counter()
        for event in root.findall(".//Event"):
            children = list(event)
            child = children[0] if children else None
            if child is None:
                continue
            event_types[child.tag] += 1
            if child.attrib.get("SyncSuccessful") == "true" and child.attrib.get("SyncedFrameId"):
                synced_types[child.tag] += 1
        rows.append({
            "match_key": asset["match_key"],
            "home": meta["HomeTeam"]["ShortName"],
            "away": meta["AwayTeam"]["ShortName"],
            "target_role": asset["target_role"],
            "target_team_code": asset["target_team_code"],
            "fps": float(md.get("framerate", meta.get("FrameRate", 50))),
            "phase_1": (md.get("phase_1_start"), md.get("phase_1_end")),
            "phase_2": (md.get("phase_2_start"), md.get("phase_2_end")),
            "players_home": len(meta["HomeTeam"].get("Players", [])),
            "players_away": len(meta["AwayTeam"].get("Players", [])),
            "kpi_events": sum(event_types.values()),
            "synced_kpi_events": sum(synced_types.values()),
            "top_synced_event_types": dict(synced_types.most_common(6)),
        })
    return pd.DataFrame(rows)


match_summary = summarize_match_files(matches)
match_summary


,match_key,home,away,target_role,target_team_code,fps,phase_1,phase_2,players_home,players_away,kpi_events,synced_kpi_events,top_synced_event_types
0,2025_09_13_FCB_HSV,FC Bayern München,Hamburger SV,home,1,50.0,"(3330943, 3484329)","(3536417, 3678119)",20,20,3306,3214,"{'Play': 1164, 'Reception': 1142, 'Carry': 338..."
1,2025_10_04_SGE_FCB,Eintracht Frankfurt,FC Bayern München,guest,0,50.0,"(3331222, 3476767)","(3526374, 3674654)",20,20,3388,3361,"{'Play': 1234, 'Reception': 1207, 'Carry': 320..."
2,2025_11_08_FCU_FCB,1. FC Union Berlin,FC Bayern München,guest,0,50.0,"(2790583, 2940975)","(2989542, 3143459)",20,20,3147,3120,"{'Play': 1065, 'Reception': 1049, 'Carry': 328..."


## Parse Bayern rosters and synced KPI events

KPI XML gives us event anchors in the same synced tracking frame space as `Positions_*.xml`. The positions file maps those synced frame counters to each half; the parquet metadata maps each half to absolute skeleton `frame_number` bounds. MatchInformation XML maps `DFL-OBJ-*` player IDs to shirt number, team, and player name. Skeleton parquet identifies players by `(team_code, jersey_number)`, so this mapping is the bridge between KPI events and 3D body motion. The candidate builder below keeps only Bayern players and clips every window to an in-play parquet phase.


In [5]:
def first_child(node):
    return next(iter(node), None)


def parse_ts(value):
    return pd.Timestamp(value) if value else pd.NaT


def xml_team_is_target(team_node) -> bool:
    if team_node.attrib.get("TeamId") == TARGET_TEAM_DFL_ID:
        return True
    values = [
        team_node.attrib.get("TeamName"),
        team_node.attrib.get("ShortName"),
        team_node.attrib.get("ThreeLetterCode"),
    ]
    values = {normalise_team_text(value) for value in values if value}
    aliases = {normalise_team_text(value) for value in TARGET_TEAM_ALIASES}
    return bool(values & aliases)


def phase_info(asset: dict) -> dict:
    meta = json.loads(Path(asset["metadata_path"]).read_text())
    md = parquet_metadata(asset["parquet_path"])
    fps = float(md.get("framerate", meta.get("FrameRate", 50)))
    out = {"fps": fps}
    for phase in range(1, 6):
        start_key = f"phase_{phase}_start"
        end_key = f"phase_{phase}_end"
        json_start = meta.get(f"Phase{phase}StartFrame", 0)
        json_end = meta.get(f"Phase{phase}EndFrame", 0)
        out[start_key] = int(float(md.get(start_key, json_start or 0)))
        out[end_key] = int(float(md.get(end_key, json_end or 0)))
    out["match_end"] = max(out[f"phase_{phase}_end"] for phase in range(1, 6))
    return out


def phase_windows(phase: dict) -> list[dict]:
    section_by_phase = {1: "firstHalf", 2: "secondHalf", 3: "extraFirstHalf", 4: "extraSecondHalf", 5: "penalties"}
    rows = []
    for phase_number in range(1, 6):
        start = int(phase.get(f"phase_{phase_number}_start", 0) or 0)
        end = int(phase.get(f"phase_{phase_number}_end", 0) or 0)
        if start > 0 and end > start:
            rows.append({
                "phase_number": phase_number,
                "phase_name": f"phase_{phase_number}",
                "game_section": section_by_phase.get(phase_number, f"phase_{phase_number}"),
                "start": start,
                "end": end,
            })
    return rows


def phase_window_for_frame(frame_number: int, phase: dict) -> dict | None:
    for window in phase_windows(phase):
        if window["start"] <= int(frame_number) <= window["end"]:
            return window
    return None


def parse_roster(matchinfo_path: Path, metadata_path: Path, match_end_frame: int) -> pd.DataFrame:
    root = ET.parse(matchinfo_path).getroot()
    meta = json.loads(Path(metadata_path).read_text())

    active_by_role_jersey = {}
    for role_key, side in [("home", "HomeTeam"), ("guest", "AwayTeam")]:
        for player in meta.get(side, {}).get("Players", []):
            start = int(player.get("StartFrameCount") or 0)
            end = int(player.get("EndFrameCount") or 0)
            if start > 0 and end <= start:
                end = match_end_frame
            active_by_role_jersey[(role_key, int(player["JerseyNo"]))] = (start, end, player)

    rows = []
    for team in root.findall(".//Team"):
        role = team.attrib.get("Role")
        team_code = 1 if role == "home" else 0 if role == "guest" else None
        is_target_team = xml_team_is_target(team)
        for player in team.findall(".//Player"):
            jersey = int(player.attrib.get("ShirtNumber", -999))
            start, end, meta_player = active_by_role_jersey.get((role, jersey), (0, 0, {}))
            rows.append({
                "player_id": player.attrib.get("PersonId"),
                "player_name": player.attrib.get("Shortname") or f"{player.attrib.get('FirstName', '')} {player.attrib.get('LastName', '')}".strip(),
                "team_code": team_code,
                "team_role": role,
                "team_name": team.attrib.get("TeamName"),
                "team_id": team.attrib.get("TeamId"),
                "is_target_team": is_target_team,
                "jersey": jersey,
                "playing_position": player.attrib.get("PlayingPosition"),
                "starting": player.attrib.get("Starting") == "true",
                "active_start": start,
                "active_end": end,
                "metadata_player_id": meta_player.get("PlayerID"),
            })
    return pd.DataFrame(rows)


def collect_person_refs(node) -> list[tuple[str, str]]:
    refs = []
    for element in node.iter():
        for attr, value in element.attrib.items():
            if isinstance(value, str) and value.startswith("DFL-OBJ"):
                refs.append((value, attr))
    seen = set()
    out = []
    for person_id, role in refs:
        if person_id not in seen:
            out.append((person_id, role))
            seen.add(person_id)
    return out


def load_phase_sync_info(positions_path: Path, phase: dict) -> tuple[pd.DataFrame, int]:
    section_to_phase = {window["game_section"]: window for window in phase_windows(phase)}
    rows = {}
    sample_numbers = []
    sample_times = []
    current_section = None

    for event, element in ET.iterparse(positions_path, events=("start", "end")):
        if event == "start" and element.tag == "FrameSet":
            current_section = element.attrib.get("GameSection")
        elif event == "start" and element.tag == "Frame" and current_section:
            frame_text = element.attrib.get("N")
            if not frame_text:
                continue
            frame_number = int(frame_text)
            frame_time = parse_ts(element.attrib.get("T"))
            rows.setdefault(current_section, {
                "game_section": current_section,
                "min_sync_frame": frame_number,
                "min_sync_time": frame_time,
            })
            if len(sample_numbers) < 10:
                sample_numbers.append(frame_number)
                sample_times.append(frame_time)
            if {"firstHalf", "secondHalf"}.issubset(rows):
                break
        elif event == "end" and element.tag == "FrameSet":
            current_section = None
            element.clear()

    phase_rows = []
    for section, row in rows.items():
        window = section_to_phase.get(section)
        if not window:
            continue
        phase_rows.append({
            **row,
            "phase_name": window["phase_name"],
            "absolute_phase_start_frame": window["start"],
            "absolute_phase_end_frame": window["end"],
        })

    deltas = []
    for index in range(1, min(len(sample_numbers), len(sample_times))):
        delta_n = sample_numbers[index] - sample_numbers[index - 1]
        delta_t = (sample_times[index] - sample_times[index - 1]).total_seconds()
        if delta_n > 0 and delta_t > 0:
            deltas.append(delta_n / delta_t)
    sync_fps = int(round(np.median(deltas))) if deltas else int(round(phase["fps"]))
    return pd.DataFrame(phase_rows).sort_values("min_sync_frame").reset_index(drop=True), sync_fps


def kpi_action_type(event_node) -> str:
    if event_node.tag == "Play":
        if (
            event_node.attrib.get("IsCross") == "true"
            and event_node.attrib.get("IsCorner") != "true"
            and event_node.attrib.get("IsFreeKick") != "true"
        ):
            return "Cross"
        flag_to_action = {
            "IsKickOff": "KickOff",
            "IsCorner": "CornerKick",
            "IsFreeKick": "FreeKick",
            "IsGoalKick": "GoalKick",
            "IsThrowIn": "ThrowIn",
        }
        for flag, action in flag_to_action.items():
            if event_node.attrib.get(flag) == "true":
                return action
        nested = first_child(event_node)
        return nested.tag if nested is not None else "Play"
    if event_node.tag == "ShotAtGoal":
        return "ShotAtGoal"
    return event_node.attrib.get("Type") or event_node.attrib.get("FoulType") or event_node.tag


def kpi_event_outcome(event_node) -> str | None:
    return (
        event_node.attrib.get("ShotResult")
        or event_node.attrib.get("Result")
        or event_node.attrib.get("Outcome")
    )


def normalized_event_text(value) -> str:
    return str(value or "").strip().casefold()


def event_matches_selection(event_type: str, action_type: str) -> bool:
    if not SELECTED_EVENT_TYPES:
        return True
    selected = {normalized_event_text(value) for value in SELECTED_EVENT_TYPES}
    return normalized_event_text(event_type) in selected


def parse_kpi_events(kpi_path: Path, phase_sync_df: pd.DataFrame, skeleton_fps: float, sync_fps: int) -> pd.DataFrame:
    root = ET.parse(kpi_path).getroot()
    phase_lookup = phase_sync_df.set_index("game_section").to_dict("index") if not phase_sync_df.empty else {}
    multiplier = float(skeleton_fps) / float(sync_fps or skeleton_fps)
    rows = []

    for event in root.findall(".//Event"):
        child = first_child(event)
        if child is None:
            continue
        if child.attrib.get("SyncSuccessful") != "true" or not child.attrib.get("SyncedFrameId"):
            continue
        action_type = kpi_action_type(child)
        if SELECTED_EVENT_TYPES and not event_matches_selection(child.tag, action_type):
            continue
        synced_frame_id = pd.to_numeric(child.attrib.get("SyncedFrameId"), errors="coerce")
        if pd.isna(synced_frame_id):
            continue
        game_section = child.attrib.get("InGameSection") or child.attrib.get("GameSection")
        phase_row = phase_lookup.get(game_section)
        if phase_row is None:
            continue
        offset_sync_frames = float(synced_frame_id) - float(phase_row["min_sync_frame"])
        absolute_frame = int(round(float(phase_row["absolute_phase_start_frame"]) + offset_sync_frames * multiplier))
        if not (int(phase_row["absolute_phase_start_frame"]) <= absolute_frame <= int(phase_row["absolute_phase_end_frame"])):
            continue

        rows.append({
            "event_id": child.attrib.get("EventId") or event.attrib.get("EventId"),
            "event_time": parse_ts(child.attrib.get("SyncedEventTime")),
            "game_time": child.attrib.get("GameTime"),
            "event_type": child.tag,
            "action_type": action_type,
            "event_outcome": kpi_event_outcome(child),
            "x": pd.to_numeric(child.attrib.get("X-Position"), errors="coerce"),
            "y": pd.to_numeric(child.attrib.get("Y-Position"), errors="coerce"),
            "person_refs": collect_person_refs(child),
            "synced_frame_id": int(round(float(synced_frame_id))),
            "frame_anchor": absolute_frame,
            "game_section": game_section,
            "phase_name": phase_row["phase_name"],
        })
    return pd.DataFrame(rows)



In [6]:
def clip_window_to_play_phase(anchor: int, phase: dict) -> dict | None:
    window = phase_window_for_frame(anchor, phase)
    if window is None:
        return None
    before = int(round(WINDOW_BEFORE_SECONDS * phase["fps"]))
    after = int(round(WINDOW_AFTER_SECONDS * phase["fps"]))
    return {
        **window,
        "frame_start": int(max(anchor - before, window["start"])),
        "frame_end": int(min(anchor + after, window["end"])),
    }


def player_active_at_frame(player, frame_number: int) -> bool:
    active_start = int(player.active_start or 0)
    active_end = int(player.active_end or 0)
    if active_start <= 0 or active_end <= active_start:
        return True
    return active_start <= int(frame_number) <= active_end


def in_play_intervals_for_player(player, phase: dict, margin: int) -> list[tuple[int, int, dict]]:
    intervals = []
    active_start = int(player.active_start or 0)
    active_end = int(player.active_end or 0)
    if active_start <= 0 or active_end <= active_start:
        active_start = 0
        active_end = phase["match_end"]

    for window in phase_windows(phase):
        lo = max(window["start"] + margin, active_start + margin)
        hi = min(window["end"] - margin, active_end - margin)
        if hi > lo:
            intervals.append((int(lo), int(hi), window))
    return intervals


def choose_anchor_from_intervals(intervals: list[tuple[int, int, dict]], rng: random.Random) -> tuple[int, dict]:
    lengths = [hi - lo + 1 for lo, hi, _ in intervals]
    pick = rng.randint(1, sum(lengths))
    total = 0
    for (lo, hi, window), length in zip(intervals, lengths):
        total += length
        if pick <= total:
            return rng.randint(lo, hi), window
    lo, hi, window = intervals[-1]
    return rng.randint(lo, hi), window


def build_candidates_for_match(asset: dict, max_event_persons: int = 3) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    phase = phase_info(asset)
    phase_sync_df, sync_fps = load_phase_sync_info(asset["positions_path"], phase)
    roster = parse_roster(asset["matchinfo_path"], asset["metadata_path"], phase["match_end"])
    if TARGET_TEAM_ONLY:
        roster_for_background = roster[roster["is_target_team"]].copy()
    else:
        roster_for_background = roster.copy()
    roster_by_player = {row.player_id: row for row in roster.itertuples(index=False)}
    events = parse_kpi_events(asset["kpi_path"], phase_sync_df, skeleton_fps=phase["fps"], sync_fps=sync_fps)
    if SELECTED_EVENT_TYPES and not events.empty:
        event_mask = events.apply(lambda row: event_matches_selection(row["event_type"], row["action_type"]), axis=1)
        events = events[event_mask].reset_index(drop=True)

    rows = []
    for event in events.itertuples(index=False):
        if event.event_type == "Delete" or not event.person_refs:
            continue
        anchor = int(event.frame_anchor)
        clipped = clip_window_to_play_phase(anchor, phase)
        if clipped is None:
            continue
        for player_id, role in event.person_refs[:max_event_persons]:
            player = roster_by_player.get(player_id)
            if player is None or player.team_code not in (0, 1):
                continue
            if TARGET_TEAM_ONLY and not bool(player.is_target_team):
                continue
            if not player_active_at_frame(player, anchor):
                continue
            rows.append({
                "match_key": asset["match_key"],
                "parquet_path": asset["parquet_path"],
                "positions_path": asset["positions_path"],
                "player_id": player_id,
                "player_name": player.player_name,
                "team_name": player.team_name,
                "team_code": int(player.team_code),
                "jersey": int(player.jersey),
                "playing_position": player.playing_position,
                "frame_anchor": anchor,
                "frame_start": clipped["frame_start"],
                "frame_end": clipped["frame_end"],
                "absolute_phase_start_frame": clipped["start"],
                "absolute_phase_end_frame": clipped["end"],
                "phase_name": clipped["phase_name"],
                "game_section": event.game_section,
                "fps": phase["fps"],
                "sync_fps": sync_fps,
                "synced_frame_id": event.synced_frame_id,
                "event_id": event.event_id,
                "event_time": event.event_time,
                "game_time": event.game_time,
                "event_type": event.event_type,
                "action_type": event.action_type,
                "event_outcome": event.event_outcome,
                "event_x": event.x,
                "event_y": event.y,
                "person_role": role,
                "candidate_source": "kpi_event",
            })

    if BACKGROUND_WINDOWS_PER_PLAYER > 0:
        stable_match_offset = sum((idx + 1) * ord(ch) for idx, ch in enumerate(asset["match_key"])) % 1000
        rng = random.Random(RANDOM_SEED + stable_match_offset)
        margin = int(round(max(WINDOW_BEFORE_SECONDS, WINDOW_AFTER_SECONDS) * phase["fps"]))
        for player in roster_for_background.itertuples(index=False):
            if player.team_code not in (0, 1):
                continue
            intervals = in_play_intervals_for_player(player, phase, margin=margin)
            if not intervals:
                continue
            for _ in range(BACKGROUND_WINDOWS_PER_PLAYER):
                anchor, window = choose_anchor_from_intervals(intervals, rng)
                clipped = clip_window_to_play_phase(anchor, phase)
                if clipped is None:
                    continue
                rows.append({
                    "match_key": asset["match_key"],
                    "parquet_path": asset["parquet_path"],
                    "positions_path": asset["positions_path"],
                    "player_id": player.player_id,
                    "player_name": player.player_name,
                    "team_name": player.team_name,
                    "team_code": int(player.team_code),
                    "jersey": int(player.jersey),
                    "playing_position": player.playing_position,
                    "frame_anchor": int(anchor),
                    "frame_start": clipped["frame_start"],
                    "frame_end": clipped["frame_end"],
                    "absolute_phase_start_frame": clipped["start"],
                    "absolute_phase_end_frame": clipped["end"],
                    "phase_name": window["phase_name"],
                    "game_section": window["game_section"],
                    "fps": phase["fps"],
                    "sync_fps": sync_fps,
                    "synced_frame_id": np.nan,
                    "event_id": None,
                    "event_time": pd.NaT,
                    "game_time": None,
                    "event_type": "background",
                    "action_type": "background",
                    "event_outcome": None,
                    "event_x": np.nan,
                    "event_y": np.nan,
                    "person_role": "background",
                    "candidate_source": "background",
                })

    candidates = pd.DataFrame(rows)
    if not candidates.empty:
        candidates = candidates.drop_duplicates(
            subset=["match_key", "player_id", "frame_start", "frame_end", "event_type"]
        ).reset_index(drop=True)
    return candidates, roster, events


all_candidates = []
all_rosters = []
all_events = []
for asset in matches.to_dict("records"):
    candidates_i, roster_i, events_i = build_candidates_for_match(asset)
    all_candidates.append(candidates_i)
    all_rosters.append(roster_i.assign(match_key=asset["match_key"]))
    all_events.append(events_i.assign(match_key=asset["match_key"]))

candidates = pd.concat(all_candidates, ignore_index=True) if all_candidates else pd.DataFrame()
rosters = pd.concat(all_rosters, ignore_index=True) if all_rosters else pd.DataFrame()
events = pd.concat(all_events, ignore_index=True) if all_events else pd.DataFrame()

print(f"Bayern candidate scenes: {len(candidates):,}")
display(candidates["event_type"].value_counts().head(12).to_frame("candidates"))
display(candidates.head())


Bayern candidate scenes: 661


,candidates
event_type,
Carry,661


,match_key,parquet_path,positions_path,player_id,player_name,team_name,team_code,jersey,playing_position,frame_anchor,frame_start,frame_end,absolute_phase_start_frame,absolute_phase_end_frame,phase_name,game_section,fps,sync_fps,synced_frame_id,event_id,event_time,game_time,event_type,action_type,event_outcome,event_x,event_y,person_role,candidate_source
0,2025_09_13_FCB_HSV,data/2025_09_13_FCB_HSV/FCB-HSV.parquet,data/2025_09_13_FCB_HSV/Positions_Bayern_Hambu...,DFL-OBJ-002GIC,J. Stanišić,FC Bayern München,1,44,LV,3331105,3330943,3331355,3330943,3484329,phase_1,firstHalf,50.0,25,10082,18902409800008,2025-09-13 16:30:20.320,000:03:24,Carry,Carry,None,23.35,-22.24,PlayerId,kpi_event
1,2025_09_13_FCB_HSV,data/2025_09_13_FCB_HSV/FCB-HSV.parquet,data/2025_09_13_FCB_HSV/Positions_Bayern_Hambu...,DFL-OBJ-0027T1,K. Laimer,FC Bayern München,1,27,RV,3331443,3331243,3331693,3330943,3484329,phase_1,firstHalf,50.0,25,10251,18902409800010,2025-09-13 16:30:27.080,000:10:00,Carry,Carry,None,30.04,23.96,PlayerId,kpi_event
2,2025_09_13_FCB_HSV,data/2025_09_13_FCB_HSV/FCB-HSV.parquet,data/2025_09_13_FCB_HSV/Positions_Bayern_Hambu...,DFL-OBJ-0000I4,Manuel Neuer,FC Bayern München,1,1,TW,3336331,3336131,3336581,3330943,3484329,phase_1,firstHalf,50.0,25,12695,18902409800028,2025-09-13 16:32:04.840,001:47:76,Carry,Carry,None,14.53,-3.01,PlayerId,kpi_event
3,2025_09_13_FCB_HSV,data/2025_09_13_FCB_HSV/FCB-HSV.parquet,data/2025_09_13_FCB_HSV/Positions_Bayern_Hambu...,DFL-OBJ-0027KL,Dayot Upamecano,FC Bayern München,1,2,IVR,3337147,3336947,3337397,3330943,3484329,phase_1,firstHalf,50.0,25,13103,18902409800037,2025-09-13 16:32:21.160,002:04:07,Carry,Carry,None,1.84,4.77,PlayerId,kpi_event
4,2025_09_13_FCB_HSV,data/2025_09_13_FCB_HSV/FCB-HSV.parquet,data/2025_09_13_FCB_HSV/Positions_Bayern_Hambu...,DFL-OBJ-0001UP,Jonathan Tah,FC Bayern München,1,4,IVL,3337803,3337603,3338053,3330943,3484329,phase_1,firstHalf,50.0,25,13431,18902409800041,2025-09-13 16:32:34.280,002:17:19,Carry,Carry,None,-2.45,-5.90,PlayerId,kpi_event


## Extract and normalize Bayern skeleton windows efficiently

The raw skeleton coordinate system is pitch-centered. For movement style, absolute location is less important than body configuration and motion, so each window is normalized by:

- sorting the 21 parts into a fixed joint order,
- interpolating missing joint values,
- centering on the pelvis,
- scaling by shoulder plus hip width,
- resampling every window to the same number of time steps,
- appending root-motion and posture summary features.

For speed and memory, full selected-event mode keeps every candidate but streams extraction one candidate window at a time with parquet frame filters. That avoids converting whole nested row groups into Python objects and avoids keeping all raw scenes in memory before featurisation.


In [7]:
def skeleton_matrix(skeleton: dict | None) -> np.ndarray:
    arr = np.full((21, 3), np.nan, dtype="float32")
    if not skeleton or not skeleton.get("parts"):
        return arr
    for part in skeleton["parts"]:
        idx = int(part["name"]) - 1
        if 0 <= idx < 21:
            arr[idx] = [part["position_x"], part["position_y"], part["position_z"]]
    return arr


def find_player_skeleton(skeletons: list, team_code: int, jersey: int) -> dict | None:
    if skeletons is None:
        return None
    for skeleton in skeletons:
        if int(skeleton.get("team", -9)) == int(team_code) and int(skeleton.get("jersey_number", -99)) == int(jersey):
            return skeleton
    return None


SCENE_PARQUET_COLUMNS = ["frame_number", "skeletons", "ball", "ball_exists"]


def row_group_frame_ranges(parquet_path: Path) -> pd.DataFrame:
    parquet_path = Path(parquet_path)
    pf = pq.ParquetFile(parquet_path)
    rows = []
    for row_group_idx in range(pf.metadata.num_row_groups):
        row_group = pf.metadata.row_group(row_group_idx)
        frame_min = None
        frame_max = None
        for col_idx in range(row_group.num_columns):
            column = row_group.column(col_idx)
            if column.path_in_schema == "frame_number" and column.statistics is not None:
                frame_min = int(column.statistics.min)
                frame_max = int(column.statistics.max)
                break
        if frame_min is None:
            table = pf.read_row_group(row_group_idx, columns=["frame_number"])
            values = table.column("frame_number").to_numpy()
            frame_min = int(values.min())
            frame_max = int(values.max())
        rows.append({"row_group": row_group_idx, "frame_min": frame_min, "frame_max": frame_max})
    return pd.DataFrame(rows)


def append_record_to_scene_buffer(buffer: dict, record: dict, candidate: pd.Series):
    skeleton = find_player_skeleton(record["skeletons"], candidate["team_code"], candidate["jersey"])
    if skeleton is None:
        return
    ball = record.get("ball") if record.get("ball_exists", record.get("ball") is not None) else None
    buffer["frames"].append(int(record["frame_number"]))
    buffer["poses"].append(skeleton_matrix(skeleton))
    buffer["balls"].append(ball)


def extract_scene_filtered(row: dict | pd.Series, dataset_obj=None, frame_stride: int = FRAME_STRIDE) -> dict | None:
    candidate = pd.Series(row.to_dict() if hasattr(row, "to_dict") else dict(row))
    dataset = dataset_obj if dataset_obj is not None else ds.dataset(Path(candidate["parquet_path"]), format="parquet")
    filt = (ds.field("frame_number") >= int(candidate["frame_start"])) & (ds.field("frame_number") <= int(candidate["frame_end"]))
    table = dataset.to_table(columns=SCENE_PARQUET_COLUMNS, filter=filt)
    buffer = {"frames": [], "poses": [], "balls": []}
    frame_stride = max(1, int(frame_stride))
    for record in table.to_pylist()[::frame_stride]:
        append_record_to_scene_buffer(buffer, record, candidate)
    return finalize_scene_buffer(buffer)


def finalize_scene_buffer(buffer: dict) -> dict | None:
    if not buffer["poses"]:
        return None
    order = np.argsort(buffer["frames"])
    return {
        "frames": np.asarray([buffer["frames"][i] for i in order], dtype=np.int64),
        "poses": np.stack([buffer["poses"][i] for i in order]),
        "balls": [buffer["balls"][i] for i in order],
    }


def extract_scenes_batched(candidate_df: pd.DataFrame, show_progress: bool = True) -> dict[int, dict]:
    work = candidate_df.copy().reset_index(drop=True)
    buffers = {idx: {"frames": [], "poses": [], "balls": []} for idx in work.index}
    scenes = {}

    for parquet_key, match_candidates in work.groupby(work["parquet_path"].map(str), sort=False):
        parquet_path = Path(parquet_key)
        pf = pq.ParquetFile(parquet_path)
        ranges = row_group_frame_ranges(parquet_path)

        row_group_to_candidate_indices = {}
        candidate_to_row_groups = {idx: [] for idx in match_candidates.index}
        for idx, candidate in match_candidates.iterrows():
            overlaps = ranges[
                (ranges["frame_max"] >= int(candidate["frame_start"])) &
                (ranges["frame_min"] <= int(candidate["frame_end"]))
            ]["row_group"].tolist()
            candidate_to_row_groups[idx] = [int(value) for value in overlaps]
            for row_group_idx in overlaps:
                row_group_to_candidate_indices.setdefault(int(row_group_idx), []).append(idx)

        dense_row_groups = {
            row_group_idx
            for row_group_idx, candidate_indices in row_group_to_candidate_indices.items()
            if len(set(candidate_indices)) >= MIN_BATCH_CANDIDATES_PER_ROW_GROUP
        }
        batched_candidate_indices = {
            idx
            for row_group_idx in dense_row_groups
            for idx in row_group_to_candidate_indices[row_group_idx]
        }

        # Dense case: read each relevant row group once and share it across overlapping candidates.
        batch_row_groups = sorted({
            row_group_idx
            for idx in batched_candidate_indices
            for row_group_idx in candidate_to_row_groups[idx]
        })
        iterator = batch_row_groups
        if show_progress and batch_row_groups:
            iterator = tqdm(iterator, desc=f"Batch row groups: {parquet_path.parent.name}", leave=False)

        for row_group_idx in iterator:
            table = pf.read_row_group(int(row_group_idx), columns=SCENE_PARQUET_COLUMNS)
            records = table.to_pylist()
            if not records:
                continue
            frame_numbers = np.fromiter((record["frame_number"] for record in records), dtype=np.int64, count=len(records))
            order = np.argsort(frame_numbers)
            frame_numbers = frame_numbers[order]
            records = [records[i] for i in order]
            candidate_indices = [idx for idx in row_group_to_candidate_indices.get(int(row_group_idx), []) if idx in batched_candidate_indices]

            for idx in candidate_indices:
                candidate = work.loc[idx]
                in_window = np.flatnonzero(
                    (frame_numbers >= int(candidate["frame_start"])) &
                    (frame_numbers <= int(candidate["frame_end"]))
                )
                for pos in in_window[::FRAME_STRIDE]:
                    append_record_to_scene_buffer(buffers[idx], records[int(pos)], candidate)

        # Sparse case: filtered reads are faster than parsing whole row groups for isolated windows.
        sparse_indices = [idx for idx in match_candidates.index if idx not in batched_candidate_indices]
        sparse_dataset = ds.dataset(parquet_path, format="parquet") if sparse_indices else None
        sparse_iterator = sparse_indices
        if show_progress and sparse_indices:
            sparse_iterator = tqdm(sparse_iterator, desc=f"Filtered windows: {parquet_path.parent.name}", leave=False)
        for idx in sparse_iterator:
            scene = extract_scene_filtered(work.loc[idx], dataset_obj=sparse_dataset)
            if scene is not None:
                scenes[idx] = scene

    for idx, buffer in buffers.items():
        scene = finalize_scene_buffer(buffer)
        if scene is not None:
            scenes[idx] = scene
    return scenes


def extract_scene(row: dict | pd.Series, frame_stride: int = FRAME_STRIDE) -> dict | None:
    return extract_scene_filtered(row, frame_stride=frame_stride)


def fill_nan_sequence(x: np.ndarray) -> np.ndarray:
    y = x.copy().reshape((x.shape[0], -1))
    for col in range(y.shape[1]):
        series = pd.Series(y[:, col])
        y[:, col] = series.interpolate(limit_direction="both").fillna(0).to_numpy()
    return y.reshape(x.shape)


def resample_time(x: np.ndarray, n_steps: int = RESAMPLE_STEPS) -> np.ndarray:
    if len(x) == n_steps:
        return x
    old = np.linspace(0, 1, len(x))
    new = np.linspace(0, 1, n_steps)
    flat = x.reshape((len(x), -1))
    out = np.vstack([np.interp(new, old, flat[:, col]) for col in range(flat.shape[1])]).T
    return out.reshape((n_steps,) + x.shape[1:])


def scene_features(scene: dict) -> tuple[np.ndarray | None, dict]:
    pose = fill_nan_sequence(scene["poses"].astype("float32"))
    if len(pose) < MIN_FRAMES_FOUND:
        return None, {}

    pelvis = pose[:, 11:12, :]
    relative = pose - pelvis
    shoulder_width = np.linalg.norm(pose[:, 3, :] - pose[:, 5, :], axis=1)
    hip_width = np.linalg.norm(pose[:, 10, :] - pose[:, 12, :], axis=1)
    scale = np.nanmedian(shoulder_width + hip_width)
    if not np.isfinite(scale) or scale < 1e-3:
        scale = 1.0
    relative = relative / scale

    relative_rs = resample_time(relative)
    velocity_rs = np.gradient(relative_rs, axis=0)

    root = pose[:, 11, :]
    root_velocity = np.diff(root[:, :2], axis=0)
    root_speed_mean = float(np.nanmean(np.linalg.norm(root_velocity, axis=1))) if len(root_velocity) else 0.0
    pose_energy = float(np.nanmean(np.linalg.norm(np.gradient(relative, axis=0), axis=2)))

    body_height = float(np.nanmedian(pose[:, 1, 2] - np.minimum(pose[:, 15, 2], pose[:, 16, 2])))
    arm_spread = float(np.nanmedian(np.linalg.norm(pose[:, 8, :] - pose[:, 9, :], axis=1)))
    torso = pose[:, 4, :] - pose[:, 11, :]
    torso_lean = float(np.nanmedian(np.linalg.norm(torso[:, :2], axis=1) / (np.abs(torso[:, 2]) + 1e-3)))
    vertical_bounce = float(np.nanstd(root[:, 2]))
    turn_amount = float(np.nanstd(np.unwrap(np.arctan2(np.gradient(root[:, 1]), np.gradient(root[:, 0])))))

    summary = np.array([
        root_speed_mean,
        pose_energy,
        body_height,
        arm_spread,
        torso_lean,
        vertical_bounce,
        turn_amount,
        scale,
    ], dtype="float32")
    feature = np.concatenate([relative_rs.ravel(), velocity_rs.ravel(), summary]).astype("float32", copy=False)
    stats = {
        "frames_found": len(pose),
        "root_speed_mean": root_speed_mean,
        "pose_energy": pose_energy,
        "motion_energy": root_speed_mean + pose_energy,
        "body_height": body_height,
        "arm_spread": arm_spread,
        "torso_lean": torso_lean,
        "vertical_bounce": vertical_bounce,
        "turn_amount": turn_amount,
        "normalization_scale": float(scale),
    }
    return feature, stats


In [8]:
def deduplicate_nearby_candidates(candidate_df: pd.DataFrame) -> pd.DataFrame:
    if candidate_df.empty:
        return candidate_df.copy()
    work = candidate_df.sort_values(
        ["event_type", "frame_anchor"],
        ascending=[True, True],
    ).reset_index(drop=True)
    kept_rows = []
    kept_anchors = {}
    for row in work.to_dict("records"):
        key = (row["match_key"], row["player_id"], row["event_type"])
        anchors = kept_anchors.setdefault(key, [])
        if any(abs(int(row["frame_anchor"]) - anchor) < MIN_CANDIDATE_GAP_FRAMES for anchor in anchors):
            continue
        kept_rows.append(row)
        anchors.append(int(row["frame_anchor"]))
    return pd.DataFrame(kept_rows)


def sample_candidates_for_runtime(candidate_df: pd.DataFrame, max_candidates: int | None = MAX_CANDIDATES) -> pd.DataFrame:
    work = deduplicate_nearby_candidates(candidate_df) if DEDUPLICATE_CANDIDATES else candidate_df.copy().reset_index(drop=True)
    if MAX_SCENES_PER_PLAYER is not None and not work.empty:
        sort_col = "motion_hint" if "motion_hint" in work.columns else "frame_anchor"
        work = (
            work.sort_values(sort_col, ascending=(sort_col == "frame_anchor"))
            .groupby(["match_key", "player_id"], sort=False)
            .head(MAX_SCENES_PER_PLAYER)
            .reset_index(drop=True)
        )
    if max_candidates is not None and len(work) > max_candidates:
        sampled_groups = []
        for _, group in work.groupby("event_type", sort=False):
            n = min(len(group), max(6, int(max_candidates * len(group) / len(work))))
            sampled_groups.append(group.sample(n=n, random_state=RANDOM_SEED))
        work = pd.concat(sampled_groups, ignore_index=True).sample(frac=1.0, random_state=RANDOM_SEED).head(max_candidates)
    return work.reset_index(drop=True)


OUTLIER_DETECTION_ENABLED = True
OUTLIER_N_NEIGHBORS = 8
OUTLIER_SCORE_QUANTILE = 0.997
OUTLIER_MAX_SCENES = 8


def detect_feature_outliers(scene_df: pd.DataFrame, X: np.ndarray) -> tuple[pd.Series, np.ndarray, float]:
    if not OUTLIER_DETECTION_ENABLED or scene_df.empty or len(scene_df) < 4:
        return pd.Series(False, index=scene_df.index), np.full(len(scene_df), np.nan), np.nan
    X_scaled = StandardScaler().fit_transform(X).astype("float32", copy=False)
    n_neighbors = min(len(scene_df), max(2, OUTLIER_N_NEIGHBORS + 1))
    distances, _ = NearestNeighbors(n_neighbors=n_neighbors).fit(X_scaled).kneighbors(X_scaled)
    outlier_scores = distances[:, 1:].mean(axis=1) if n_neighbors > 1 else np.zeros(len(scene_df))
    threshold = float(np.nanquantile(outlier_scores, OUTLIER_SCORE_QUANTILE))
    mask_values = outlier_scores >= threshold
    if OUTLIER_MAX_SCENES is not None and int(mask_values.sum()) > int(OUTLIER_MAX_SCENES):
        keep_indices = np.argsort(outlier_scores)[-int(OUTLIER_MAX_SCENES):]
        capped = np.zeros(len(scene_df), dtype=bool)
        capped[keep_indices] = True
        mask_values = capped
    return pd.Series(mask_values, index=scene_df.index), outlier_scores, threshold


def remove_scene_outliers(scene_df: pd.DataFrame, X: np.ndarray) -> tuple[pd.DataFrame, np.ndarray, pd.DataFrame]:
    mask, outlier_scores, threshold = detect_feature_outliers(scene_df, X)
    work = scene_df.copy()
    work["outlier_score"] = outlier_scores
    work["outlier_threshold"] = threshold
    work["outlier_reason"] = "feature_knn_distance"
    removed = work.loc[mask].sort_values("outlier_score", ascending=False).copy()
    if removed.empty:
        return work.reset_index(drop=True), X, removed
    removed["outlier_rank"] = np.arange(1, len(removed) + 1)
    keep = (~mask).to_numpy()
    return work.loc[~mask].reset_index(drop=True), X[keep], removed.reset_index(drop=True)


def build_feature_matrix(candidates: pd.DataFrame, max_candidates: int | None = MAX_CANDIDATES):
    work = sample_candidates_for_runtime(candidates, max_candidates=max_candidates)
    if work.empty:
        raise RuntimeError("No candidate windows were generated for the selected event filter.")

    rows = []
    features = []
    failures = []
    datasets = {}
    iterator = tqdm(list(work.iterrows()), desc="Extracting and featurising Bayern windows")
    for idx, row in iterator:
        parquet_key = str(row["parquet_path"])
        dataset_obj = datasets.get(parquet_key)
        if dataset_obj is None:
            dataset_obj = ds.dataset(Path(row["parquet_path"]), format="parquet")
            datasets[parquet_key] = dataset_obj
        scene = extract_scene_filtered(row, dataset_obj=dataset_obj, frame_stride=FRAME_STRIDE)
        if scene is None:
            failures.append((row.to_dict(), "no skeleton rows"))
            continue
        feature, stats = scene_features(scene)
        if feature is None:
            failures.append((row.to_dict(), "too few frames"))
            continue
        rows.append({**row.to_dict(), **stats})
        features.append(feature)

    if not features:
        raise RuntimeError("No usable Bayern skeleton windows were extracted. Check paths, parquet engine, or frame mapping.")
    scene_df = pd.DataFrame(rows)
    X = np.vstack(features).astype("float32", copy=False)
    return scene_df, X, failures


scene_df, X, failures = build_feature_matrix(candidates)
scene_df, X, removed_outlier_scenes = remove_scene_outliers(scene_df, X)
if not removed_outlier_scenes.empty:
    print(f"Removed {len(removed_outlier_scenes):,} automatic outlier scene(s) before PCA/ranking.")
    print(f"Outlier animations will be rendered under {OUTPUT_DIR / 'outliers'} when the MP4 render cell runs.")
    display(removed_outlier_scenes[["outlier_rank", "player_name", "event_type", "event_id", "match_key", "game_time", "frame_start", "frame_end", "outlier_score"]])
print(f"Usable Bayern scenes: {len(scene_df):,} | feature dimension: {X.shape[1]:,} | failures: {len(failures):,}")
scene_df.head()


Extracting and featurising Bayern windows:   0%|          | 0/661 [00:00<?, ?it/s]

Removed 2 automatic outlier scene(s) before PCA/ranking.
Outlier animations will be rendered under signature_scene_outputs/outliers when the MP4 render cell runs.


,outlier_rank,player_name,event_type,event_id,match_key,game_time,frame_start,frame_end,outlier_score
0,1,Michael Olise,Carry,18909309801040,2025_11_08_FCU_FCB,061:43:96,3039540,3039990,653.496552
1,2,K. Laimer,Carry,18909309800708,2025_11_08_FCU_FCB,046:52:84,2931025,2931475,264.933655


Usable Bayern scenes: 658 | feature dimension: 3,032 | failures: 1


,match_key,parquet_path,positions_path,player_id,player_name,team_name,team_code,jersey,playing_position,frame_anchor,frame_start,frame_end,absolute_phase_start_frame,absolute_phase_end_frame,phase_name,game_section,fps,sync_fps,synced_frame_id,event_id,event_time,game_time,event_type,action_type,event_outcome,event_x,event_y,person_role,candidate_source,frames_found,root_speed_mean,pose_energy,motion_energy,body_height,arm_spread,torso_lean,vertical_bounce,turn_amount,normalization_scale,outlier_score,outlier_threshold,outlier_reason
0,2025_09_13_FCB_HSV,data/2025_09_13_FCB_HSV/FCB-HSV.parquet,data/2025_09_13_FCB_HSV/Positions_Bayern_Hambu...,DFL-OBJ-002GIC,J. Stanišić,FC Bayern München,1,44,LV,3331105,3330943,3331355,3330943,3484329,phase_1,firstHalf,50.0,25,10082,18902409800008,2025-09-13 16:30:20.320,000:03:24,Carry,Carry,None,23.35,-22.24,PlayerId,kpi_event,52,0.678868,0.309997,0.988864,1.40905,0.560630,0.253715,0.045012,1.310513,0.485025,54.260182,98.202775,feature_knn_distance
1,2025_09_13_FCB_HSV,data/2025_09_13_FCB_HSV/FCB-HSV.parquet,data/2025_09_13_FCB_HSV/Positions_Bayern_Hambu...,DFL-OBJ-0027T1,K. Laimer,FC Bayern München,1,27,RV,3331443,3331243,3331693,3330943,3484329,phase_1,firstHalf,50.0,25,10251,18902409800010,2025-09-13 16:30:27.080,000:10:00,Carry,Carry,None,30.04,23.96,PlayerId,kpi_event,57,0.765990,0.273824,1.039814,1.34540,0.684665,0.305177,0.088345,1.063083,0.518103,54.082345,98.202775,feature_knn_distance
2,2025_09_13_FCB_HSV,data/2025_09_13_FCB_HSV/FCB-HSV.parquet,data/2025_09_13_FCB_HSV/Positions_Bayern_Hambu...,DFL-OBJ-0000I4,Manuel Neuer,FC Bayern München,1,1,TW,3336331,3336131,3336581,3330943,3484329,phase_1,firstHalf,50.0,25,12695,18902409800028,2025-09-13 16:32:04.840,001:47:76,Carry,Carry,None,14.53,-3.01,PlayerId,kpi_event,57,0.302167,0.172804,0.474971,1.59240,0.528571,0.119014,0.050502,1.207827,0.546476,48.034253,98.202775,feature_knn_distance
3,2025_09_13_FCB_HSV,data/2025_09_13_FCB_HSV/FCB-HSV.parquet,data/2025_09_13_FCB_HSV/Positions_Bayern_Hambu...,DFL-OBJ-0027KL,Dayot Upamecano,FC Bayern München,1,2,IVR,3337147,3336947,3337397,3330943,3484329,phase_1,firstHalf,50.0,25,13103,18902409800037,2025-09-13 16:32:21.160,002:04:07,Carry,Carry,None,1.84,4.77,PlayerId,kpi_event,57,0.267280,0.187696,0.454976,1.48380,0.590244,0.177588,0.068656,1.358236,0.543541,44.928558,98.202775,feature_knn_distance
4,2025_09_13_FCB_HSV,data/2025_09_13_FCB_HSV/FCB-HSV.parquet,data/2025_09_13_FCB_HSV/Positions_Bayern_Hambu...,DFL-OBJ-0001UP,Jonathan Tah,FC Bayern München,1,4,IVL,3337803,3337603,3338053,3330943,3484329,phase_1,firstHalf,50.0,25,13431,18902409800041,2025-09-13 16:32:34.280,002:17:19,Carry,Carry,None,-2.45,-5.90,PlayerId,kpi_event,57,0.606118,0.264233,0.870351,1.46060,0.605323,0.238848,0.072571,0.952612,0.539563,50.759644,98.202775,feature_knn_distance


## Score signature scenes

The scoring is intentionally transparent:

- `distinctiveness_ratio`: nearest other-player distance divided by nearest same-player distance. Higher means the scene sits closer to the player's own movement family than to other players.
- `repeatability`: inverse nearest same-player distance. Higher means this motion has a same-player echo elsewhere.
- `motion_energy`: root movement plus pose velocity.

The final score is a weighted rank blend. This avoids over-trusting any single raw scale.


In [9]:
def rank01(series: pd.Series) -> pd.Series:
    return series.replace([np.inf, -np.inf], np.nan).fillna(series.median()).rank(pct=True)


def text_value(value, fallback: str = "") -> str:
    if value is None:
        return fallback
    try:
        if pd.isna(value):
            return fallback
    except (TypeError, ValueError):
        pass
    text = str(value).strip()
    return text if text else fallback


def ranking_event_type_label(row: dict) -> str:
    event = text_value(row.get("event_type"), "")
    action = text_value(row.get("action_type"), "")
    if event and event != "background":
        return event
    if action and action != "background":
        return action
    return "open_play_movement"


def score_signature_scenes(scene_df: pd.DataFrame, X: np.ndarray):
    X_scaled = StandardScaler().fit_transform(X).astype("float32", copy=False)
    n_components = max(2, min(32, X_scaled.shape[0] - 1, X_scaled.shape[1]))
    reducer = PCA(n_components=n_components, random_state=RANDOM_SEED)
    embedding = reducer.fit_transform(X_scaled)

    n_neighbors = min(16, len(scene_df))
    nbrs = NearestNeighbors(n_neighbors=n_neighbors).fit(embedding)
    distances, indices = nbrs.kneighbors(embedding)

    player_keys = scene_df["player_id"].astype(str).to_numpy()
    global_default_distance = float(np.nanmedian(distances[:, 1:])) if n_neighbors > 1 else 1.0

    same_min = []
    other_min = []
    local_density = []
    for i in range(len(scene_df)):
        same_distances = []
        other_distances = []
        for distance, neighbor_idx in zip(distances[i, 1:], indices[i, 1:]):
            if player_keys[neighbor_idx] == player_keys[i]:
                same_distances.append(distance)
            else:
                other_distances.append(distance)
        same_min.append(min(same_distances) if same_distances else global_default_distance)
        other_min.append(min(other_distances) if other_distances else global_default_distance)
        local_density.append(float(np.mean(distances[i, 1:])) if n_neighbors > 1 else 0.0)

    scored = scene_df.copy()
    scored["same_player_nn_distance"] = same_min
    scored["other_player_nn_distance"] = other_min
    scored["distinctiveness_ratio"] = scored["other_player_nn_distance"] / (scored["same_player_nn_distance"] + 1e-6)
    scored["repeatability"] = 1.0 / (1.0 + scored["same_player_nn_distance"])
    scored["local_density_distance"] = local_density
    for component_idx in range(min(3, embedding.shape[1])):
        scored[f"pca_{component_idx + 1}"] = embedding[:, component_idx]
    scored["embedding_x"] = scored["pca_1"]
    scored["embedding_y"] = scored["pca_2"]
    scored["ranking_event_type"] = scored.apply(ranking_event_type_label, axis=1)

    scored["signature_score"] = (
        0.45 * rank01(scored["distinctiveness_ratio"]) +
        0.30 * rank01(scored["repeatability"]) +
        0.25 * rank01(scored["motion_energy"])
    )
    scored["reason"] = scored.apply(
        lambda r: (
            f"{r['event_type']}; distinctive={r['distinctiveness_ratio']:.2f}; "
            f"repeatable={r['repeatability']:.2f}; "
            f"motion={r['motion_energy']:.2f}"
        ),
        axis=1,
    )
    return scored.sort_values("signature_score", ascending=False).reset_index(drop=True), embedding, reducer


ranked_scenes, embedding, reducer = score_signature_scenes(scene_df, X)

cols = [
    "signature_score", "match_key", "player_name", "team_name", "jersey", "playing_position",
    "ranking_event_type", "event_type", "action_type", "event_outcome", "game_time", "person_role", "frame_start", "frame_end", "frame_anchor",
    "pca_1", "pca_2", "distinctiveness_ratio", "repeatability", "motion_energy", "reason",
]
ranked_scenes[cols].head(25)


,signature_score,match_key,player_name,team_name,jersey,playing_position,ranking_event_type,event_type,action_type,event_outcome,game_time,person_role,frame_start,frame_end,frame_anchor,pca_1,pca_2,distinctiveness_ratio,repeatability,motion_energy,reason
0,0.915805,2025_11_08_FCU_FCB,Jonathan Tah,FC Bayern München,4,IVL,Carry,Carry,Carry,None,023:04:59,PlayerId,2859613,2860063,2859813,-15.259616,-8.382329,1.435492,0.047136,0.856838,Carry; distinctive=1.44; repeatable=0.05; moti...
1,0.902660,2025_09_13_FCB_HSV,Joshua Kimmich,FC Bayern München,6,DMR,Carry,Carry,Carry,None,046:39:40,PlayerId,3470713,3471163,3470913,-9.625772,8.238953,1.693852,0.060956,0.781053,Carry; distinctive=1.69; repeatable=0.06; moti...
2,0.901368,2025_09_13_FCB_HSV,Joshua Kimmich,FC Bayern München,6,DMR,Carry,Carry,Carry,None,004:51:92,PlayerId,3345339,3345789,3345539,-10.672716,6.247397,1.557419,0.060956,0.782038,Carry; distinctive=1.56; repeatable=0.06; moti...
3,0.891793,2025_11_08_FCU_FCB,Michael Olise,FC Bayern München,17,ORM,Carry,Carry,Carry,None,089:19:52,PlayerId,3122318,3122768,3122518,18.364269,10.818527,1.270505,0.044552,0.844865,Carry; distinctive=1.27; repeatable=0.04; moti...
4,0.882903,2025_09_13_FCB_HSV,Jonathan Tah,FC Bayern München,4,IVL,Carry,Carry,Carry,None,050:31:67,PlayerId,3552801,3553251,3553001,20.028727,-2.529679,1.231547,0.035964,1.063549,Carry; distinctive=1.23; repeatable=0.04; moti...
5,0.877508,2025_11_08_FCU_FCB,Luis Díaz,FC Bayern München,14,OLM,Carry,Carry,Carry,None,083:20:47,PlayerId,3104366,3104816,3104566,6.037315,-27.193924,1.443873,0.046928,0.769290,Carry; distinctive=1.44; repeatable=0.05; moti...
6,0.875608,2025_11_08_FCU_FCB,Jonathan Tah,FC Bayern München,4,IVL,Carry,Carry,Carry,None,022:22:51,PlayerId,2857509,2857959,2857709,-7.599746,-3.772420,1.243021,0.048993,0.799166,Carry; distinctive=1.24; repeatable=0.05; moti...
7,0.874620,2025_11_08_FCU_FCB,Jonathan Tah,FC Bayern München,4,IVL,Carry,Carry,Carry,None,076:48:23,PlayerId,3084754,3085204,3084954,18.853701,1.334709,1.129854,0.042028,0.919479,Carry; distinctive=1.13; repeatable=0.04; moti...
8,0.871657,2025_11_08_FCU_FCB,K. Laimer,FC Bayern München,27,RV,Carry,Carry,Carry,None,065:47:23,PlayerId,3051704,3052154,3051904,22.098694,1.585081,1.112161,0.037614,1.251835,Carry; distinctive=1.11; repeatable=0.04; moti...
9,0.853571,2025_09_13_FCB_HSV,Jonathan Tah,FC Bayern München,4,IVL,Carry,Carry,Carry,None,057:15:11,PlayerId,3572973,3573423,3573173,17.377653,9.585293,1.121323,0.053215,0.804448,Carry; distinctive=1.12; repeatable=0.05; moti...


In [10]:
# Render the top-ranked signature scenes for each Bayern player and top-level event.
ANIMATION_DIR = OUTPUT_DIR / "player_signature_animations"
OUTLIER_ANIMATION_DIR = OUTPUT_DIR / "outliers"
ANIMATION_DIR.mkdir(parents=True, exist_ok=True)
OUTLIER_ANIMATION_DIR.mkdir(parents=True, exist_ok=True)

MP4S_PER_PLAYER_EVENT = 3
MP4_RENDER_FPS = None  # None keeps the MP4 in real time using the source skeleton FPS.
MP4_RENDER_FRAME_STRIDE = 1  # Use every raw frame for the final animation.
MP4_FOLLOW_PLAYER = True  # Focus x/y around the player's pelvis, like goal_moment_animations.ipynb.
MP4_PLAYER_FOCUS_RADIUS = 2.0  # meters around the player in x/y, matching the fixed-view animation setup.
MP4_DPI = 140
MP4_BITRATE = 2200
MP4_CONTEXT_BEFORE_SECONDS = 4.0
MP4_CONTEXT_AFTER_SECONDS = 5.0
OVERWRITE_ANIMATIONS = True
CLEAN_ANIMATION_DIR = True
CLEAN_OUTLIER_ANIMATION_DIR = True
DIVERSIFY_PLAYER_TOP_SCENES = True
DIVERSITY_MIN_SCENE_GAP_SECONDS = MP4_CONTEXT_BEFORE_SECONDS + MP4_CONTEXT_AFTER_SECONDS
DIVERSITY_MIN_PCA_DISTANCE_QUANTILE = 0.20

metadata_cols = [
    "signature_score", "player_event_scene_rank", "player_scene_rank", "match_key", "player_id", "player_name", "team_name", "team_code", "jersey",
    "playing_position", "ranking_event_type", "player_event_type_count", "selection_note", "event_type", "action_type", "event_outcome", "event_id", "event_time", "game_time", "event_x", "event_y",
    "frame_start", "frame_end", "frame_anchor", "absolute_phase_start_frame", "absolute_phase_end_frame",
    "phase_name", "game_section", "synced_frame_id", "sync_fps", "positions_path",
    "candidate_source", "person_role", "fps",
    "distinctiveness_ratio", "repeatability", "motion_energy",
    "root_speed_mean", "pose_energy", "body_height", "arm_spread", "torso_lean", "vertical_bounce", "turn_amount",
    "outlier_rank", "outlier_score", "outlier_threshold", "outlier_reason",
    "reason",
]


def safe_filename(value) -> str:
    safe = "".join(ch.lower() if ch.isalnum() else "_" for ch in str(value)).strip("_")
    while "__" in safe:
        safe = safe.replace("__", "_")
    return safe or "scene"


def row_event_count_label(row: dict) -> str:
    label = row.get("ranking_event_type")
    if not is_missing_value(label):
        return display_value(label, "open_play_movement")
    event = display_value(row.get("event_type"), "")
    action = display_value(row.get("action_type"), "")
    if event and event != "background":
        return event
    if action and action != "background":
        return action
    return "open_play_movement"


def player_animation_dir_name(row: dict) -> str:
    player = safe_filename(display_value(row.get("player_name"), "player"))
    count = row.get("player_event_type_count")
    if is_missing_value(count):
        return f"{player}_unknown_events"
    count = int(count)
    unit = "event" if count == 1 else "events"
    return f"{player}_{count}_{unit}"


def animation_file_name(row: dict) -> str:
    event_label = safe_filename(row_event_count_label(row))
    rank = row.get("player_event_scene_rank", row.get("player_scene_rank"))
    if is_missing_value(rank):
        return f"{event_label}_preview.mp4"
    return f"{event_label}_rank_{int(rank):02d}.mp4"


def animation_output_path(row: dict) -> Path:
    player_dir = ANIMATION_DIR / player_animation_dir_name(row)
    return player_dir / animation_file_name(row)


def format_number(value, digits: int = 3) -> str:
    if pd.isna(value):
        return "n/a"
    return f"{float(value):.{digits}f}"


def is_missing_value(value) -> bool:
    if value is None:
        return True
    try:
        return bool(pd.isna(value))
    except (TypeError, ValueError):
        return False


def display_value(value, fallback: str = "n/a") -> str:
    if is_missing_value(value):
        return fallback
    text = str(value).strip()
    return text if text else fallback


BALL_TRACK_CACHE = {}


def numeric_value(value, fallback=np.nan) -> float:
    parsed = pd.to_numeric(value, errors="coerce")
    return fallback if pd.isna(parsed) else float(parsed)


def positions_path_for_row(row: dict) -> Path | None:
    path_value = row.get("positions_path")
    if not is_missing_value(path_value):
        path = Path(path_value)
        if path.exists():
            return path
    match_key = display_value(row.get("match_key"), "")
    if not match_key:
        return None
    return find_single(DATA_DIR / match_key, "Positions_*.xml")


def load_ball_track(positions_path: Path) -> pd.DataFrame:
    path = Path(positions_path)
    cache_key = str(path.resolve())
    if cache_key in BALL_TRACK_CACHE:
        return BALL_TRACK_CACHE[cache_key]

    rows = []
    current_section = None
    in_ball_frameset = False
    for event, element in ET.iterparse(path, events=("start", "end")):
        if event == "start" and element.tag == "FrameSet":
            current_section = element.attrib.get("GameSection")
            in_ball_frameset = (
                element.attrib.get("TeamId") == "BALL"
                or element.attrib.get("PersonId") == "DFL-OBJ-0000XT"
            )
        elif event == "start" and element.tag == "Frame" and in_ball_frameset:
            frame_number = pd.to_numeric(element.attrib.get("N"), errors="coerce")
            if not pd.isna(frame_number):
                z_value = numeric_value(element.attrib.get("Z"), fallback=0.11)
                rows.append({
                    "game_section": current_section,
                    "synced_frame_id": int(frame_number),
                    "frame_time": parse_ts(element.attrib.get("T")),
                    "ball_x": numeric_value(element.attrib.get("X")),
                    "ball_y": numeric_value(element.attrib.get("Y")),
                    "ball_z": z_value,
                    "ball_status": element.attrib.get("BallStatus"),
                    "ball_possession": element.attrib.get("BallPossession"),
                })
        elif event == "end":
            if element.tag == "FrameSet":
                current_section = None
                in_ball_frameset = False
            element.clear()

    ball_df = pd.DataFrame(rows)
    if not ball_df.empty:
        ball_df = ball_df.sort_values(["game_section", "synced_frame_id"]).reset_index(drop=True)
    BALL_TRACK_CACHE[cache_key] = ball_df
    return ball_df


def infer_sync_fps(ball_df: pd.DataFrame, fallback: float = 25.0) -> float:
    if ball_df.empty or "frame_time" not in ball_df.columns:
        return fallback
    sample = ball_df.dropna(subset=["frame_time"]).head(80)
    if len(sample) < 2:
        return fallback
    frame_deltas = sample["synced_frame_id"].diff().to_numpy(dtype="float64")
    time_deltas = sample["frame_time"].diff().dt.total_seconds().to_numpy(dtype="float64")
    valid = (frame_deltas > 0) & (time_deltas > 0)
    if not valid.any():
        return fallback
    return float(np.nanmedian(frame_deltas[valid] / time_deltas[valid]))


def parquet_ball_positions_for_scene(scene_balls, expected_length: int) -> np.ndarray:
    positions = np.full((expected_length, 3), np.nan, dtype="float32")
    if scene_balls is None:
        return positions
    for idx, ball in enumerate(list(scene_balls)[:expected_length]):
        if not isinstance(ball, dict):
            continue
        xyz = [
            numeric_value(ball.get("position_x")),
            numeric_value(ball.get("position_y")),
            numeric_value(ball.get("position_z")),
        ]
        if np.all(np.isfinite(xyz)):
            positions[idx] = np.asarray(xyz, dtype="float32")
    return positions


def ball_positions_for_frames(row: dict, frames: np.ndarray, scene_balls=None) -> np.ndarray:
    empty = np.full((len(frames), 3), np.nan, dtype="float32")
    parquet_positions = parquet_ball_positions_for_scene(scene_balls, len(frames))
    if np.any(np.all(np.isfinite(parquet_positions), axis=1)):
        return parquet_positions

    positions_path = positions_path_for_row(row)
    if positions_path is None:
        return empty

    synced_anchor = row.get("synced_frame_id")
    frame_anchor = row.get("frame_anchor")
    if is_missing_value(synced_anchor) or is_missing_value(frame_anchor):
        return empty

    ball_df = load_ball_track(positions_path)
    if ball_df.empty:
        return empty
    game_section = display_value(row.get("game_section"), "")
    if game_section:
        section_df = ball_df[ball_df["game_section"] == game_section]
        if section_df.empty:
            section_df = ball_df
    else:
        section_df = ball_df

    skeleton_fps = float(row.get("fps", 50) or 50)
    sync_fps = row.get("sync_fps")
    sync_fps = infer_sync_fps(section_df) if is_missing_value(sync_fps) else float(sync_fps)
    target_sync_frames = np.rint(
        float(synced_anchor) + (frames.astype("float64") - float(frame_anchor)) * sync_fps / skeleton_fps
    ).astype("int64")

    track = section_df.drop_duplicates("synced_frame_id").set_index("synced_frame_id")
    aligned = track.reindex(target_sync_frames)[["ball_x", "ball_y", "ball_z"]].to_numpy(dtype="float32")
    return aligned


def scene_duration_seconds(row: dict) -> float:
    fps = float(row.get("fps", 50) or 50)
    start = row.get("frame_start")
    end = row.get("frame_end")
    if is_missing_value(start) or is_missing_value(end):
        return 0.0
    return max(0.0, (float(end) - float(start)) / fps)


def scene_event_label(row: dict) -> str:
    event = display_value(row.get("event_type"), "")
    action = display_value(row.get("action_type"), "")
    if event and event != "background":
        return event
    if action and action != "background":
        return action
    return "Open play movement"


def scene_event_outcome_label(row: dict) -> str:
    outcome = display_value(row.get("event_outcome"), "")
    if not outcome or outcome == "background":
        return ""
    readable = "".join(f" {ch.lower()}" if ch.isupper() else ch for ch in outcome).strip()
    return f"outcome {readable}"


def scene_metadata_text(row: dict) -> str:
    player = display_value(row.get("player_name"))
    jersey = row.get("jersey")
    jersey_text = f"#{int(jersey)}" if not is_missing_value(jersey) else ""
    position = display_value(row.get("playing_position"), "")
    match = display_value(row.get("match_key"), "Match")
    game_time = display_value(row.get("game_time"), "")
    duration = scene_duration_seconds(row)

    line_one = "  |  ".join(part for part in [player, jersey_text, position] if part)
    line_two_parts = [
        scene_event_label(row),
        scene_event_outcome_label(row),
        match,
        f"match time {game_time}" if game_time else "",
        f"{duration:.1f}s clip",
        f"score {format_number(row.get('signature_score'), 2)}",
    ]
    line_two = "  |  ".join(part for part in line_two_parts if part)
    lines = [line_one, line_two]
    return "\n".join(lines)


def render_context_row(row: dict) -> dict:
    render_row = dict(row)
    anchor_value = render_row.get("frame_anchor")
    if is_missing_value(anchor_value):
        return render_row

    skeleton_fps = float(render_row.get("fps", 50) or 50)
    anchor = int(round(float(anchor_value)))
    start = anchor - int(round(MP4_CONTEXT_BEFORE_SECONDS * skeleton_fps))
    end = anchor + int(round(MP4_CONTEXT_AFTER_SECONDS * skeleton_fps))

    phase_start = render_row.get("absolute_phase_start_frame")
    phase_end = render_row.get("absolute_phase_end_frame")
    start = max(int(phase_start), start) if not is_missing_value(phase_start) else max(0, start)
    end = min(int(phase_end), end) if not is_missing_value(phase_end) else end

    if end > start:
        render_row["frame_start"] = int(start)
        render_row["frame_end"] = int(end)
    return render_row


def render_scene_mp4(scene_row, output_path: Path | None = None, fps: int | None = MP4_RENDER_FPS):
    import os

    mpl_config_dir = Path("/tmp/matplotlib")
    mpl_config_dir.mkdir(parents=True, exist_ok=True)
    os.environ.setdefault("MPLCONFIGDIR", str(mpl_config_dir))

    import matplotlib.pyplot as plt
    from matplotlib.animation import FuncAnimation, FFMpegWriter

    row = scene_row.to_dict() if hasattr(scene_row, "to_dict") else dict(scene_row)
    row = render_context_row(row)
    scene = extract_scene(row, frame_stride=MP4_RENDER_FRAME_STRIDE)
    if scene is None:
        raise ValueError("No skeleton data found for this scene")
    poses = scene["poses"].astype("float32")
    frames = scene["frames"]
    ball_positions = ball_positions_for_frames(row, frames, scene.get("balls"))

    if output_path is None:
        output_path = animation_output_path(row)
    else:
        output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    valid_points = poses.reshape(-1, 3)
    valid_points = valid_points[np.all(np.isfinite(valid_points), axis=1)]
    if len(valid_points) == 0:
        raise ValueError("No finite raw joint coordinates found for this scene")
    valid_pelvis = poses[:, 11, :]
    valid_pelvis = valid_pelvis[np.all(np.isfinite(valid_pelvis), axis=1)]
    fixed_center = np.nanmedian(valid_pelvis, axis=0) if len(valid_pelvis) else np.nanmedian(valid_points, axis=0)
    focus_radius = float(MP4_PLAYER_FOCUS_RADIUS)
    fixed_zlim = (0.0, 2.5)
    axis_box_aspect = (1.0, 1.0, 1.0)

    metadata_text = scene_metadata_text(row)
    fig = plt.figure(figsize=(8, 8), facecolor="white")
    fig.subplots_adjust(left=0.02, right=0.98, bottom=0.02, top=0.84)
    fig.text(0.02, 0.985, metadata_text, ha="left", va="top", fontsize=10, linespacing=1.35)
    ax = fig.add_subplot(111, projection="3d")

    def style_white_axis():
        ax.set_facecolor("white")
        for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
            axis.pane.set_facecolor((1, 1, 1, 1))
            axis.pane.set_edgecolor((0.85, 0.85, 0.85, 1))

    def focus_center_for_pose(pose: np.ndarray) -> np.ndarray:
        pelvis = pose[11]
        if MP4_FOLLOW_PLAYER and np.all(np.isfinite(pelvis)):
            return pelvis
        return fixed_center

    anchor = int(row.get("frame_anchor", frames[len(frames) // 2]))
    skeleton_fps = float(row.get("fps", 50) or 50)
    render_fps = int(round(fps or skeleton_fps / max(1, int(MP4_RENDER_FRAME_STRIDE))))

    def update(i):
        ax.clear()
        style_white_axis()
        pose = poses[i]
        for a, b in BONES:
            p1, p2 = pose[a - 1], pose[b - 1]
            if not (np.all(np.isfinite(p1)) and np.all(np.isfinite(p2))):
                continue
            ax.plot([p1[0], p2[0]], [p1[1], p2[1]], [p1[2], p2[2]], color="#1f77b4", linewidth=2.8)
        finite_joints = np.all(np.isfinite(pose), axis=1)
        if finite_joints.any():
            visible_pose = pose[finite_joints]
            ax.scatter(visible_pose[:, 0], visible_pose[:, 1], visible_pose[:, 2], color="#d62728", s=20)
        ball_xyz = ball_positions[i]
        if np.all(np.isfinite(ball_xyz)):
            trail = ball_positions[max(0, i - 15):i + 1]
            trail = trail[np.all(np.isfinite(trail), axis=1)]
            if len(trail) > 1:
                ax.plot(trail[:, 0], trail[:, 1], trail[:, 2], color="#f2a900", linewidth=2.0, alpha=0.75)
            ax.scatter(
                [ball_xyz[0]], [ball_xyz[1]], [ball_xyz[2]],
                color="#f2a900", edgecolors="#111111", linewidths=0.6, s=70, depthshade=True,
            )
        focus_center = focus_center_for_pose(pose)
        ax.set_xlim(focus_center[0] - focus_radius, focus_center[0] + focus_radius)
        ax.set_ylim(focus_center[1] - focus_radius, focus_center[1] + focus_radius)
        ax.set_zlim(*fixed_zlim)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_zticks([])
        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.set_zlabel("")
        ax.grid(True, alpha=0.25)
        ax.set_box_aspect(axis_box_aspect)
        seconds = (int(frames[i]) - anchor) / skeleton_fps
        ax.set_title(f"frame {int(frames[i])}  |  t={seconds:+.2f}s from anchor", pad=8, fontsize=10)
        ax.view_init(elev=18, azim=-65)

    anim = FuncAnimation(fig, update, frames=len(poses), interval=1000 / render_fps, repeat=False)
    writer = FFMpegWriter(
        fps=render_fps,
        bitrate=MP4_BITRATE,
        metadata={
            "title": f"{row.get('player_name')} signature scene",
            "artist": "signature_scene_selection.ipynb",
            "comment": metadata_text.replace("\n", " | "),
        },
    )
    anim.save(output_path, writer=writer, dpi=MP4_DPI)
    plt.close(fig)
    return output_path


def outlier_animation_output_path(row: dict) -> Path:
    rank = row.get("outlier_rank")
    rank_text = f"rank_{int(rank):02d}" if not is_missing_value(rank) else "outlier"
    player = safe_filename(display_value(row.get("player_name"), "player"))
    event_label = safe_filename(row_event_count_label(row))
    event_id = display_value(row.get("event_id"), "")
    if event_id:
        scene_key = safe_filename(event_id)
    else:
        scene_key = safe_filename(f"{row.get('match_key')}_{row.get('frame_anchor')}")
    return OUTLIER_ANIMATION_DIR / f"{rank_text}_{player}_{event_label}_{scene_key}.mp4"


def render_outlier_mp4s(outlier_df: pd.DataFrame) -> pd.DataFrame:
    if outlier_df is None or outlier_df.empty:
        return pd.DataFrame()
    if CLEAN_OUTLIER_ANIMATION_DIR:
        for old_file in OUTLIER_ANIMATION_DIR.glob("*.mp4"):
            old_file.unlink()
    results = []
    iterator = tqdm(list(outlier_df.iterrows()), desc="Rendering outlier MP4s")
    for _, row in iterator:
        row_for_render = render_context_row(row.to_dict())
        row_for_render.setdefault("ranking_event_type", row_event_count_label(row_for_render))
        output_file = outlier_animation_output_path(row_for_render)
        if output_file.exists() and not OVERWRITE_ANIMATIONS:
            results.append({**{col: row_for_render.get(col) for col in metadata_cols if col in row_for_render}, "output_path": str(output_file), "status": "exists"})
            continue
        try:
            rendered_path = render_scene_mp4(row_for_render, output_file)
            results.append({**{col: row_for_render.get(col) for col in metadata_cols if col in row_for_render}, "output_path": str(rendered_path), "status": "rendered"})
        except Exception as exc:
            results.append({**{col: row_for_render.get(col) for col in metadata_cols if col in row_for_render}, "output_path": str(output_file), "status": "failed", "error": str(exc)})
    return pd.DataFrame(results)


outlier_animation_results = render_outlier_mp4s(removed_outlier_scenes if "removed_outlier_scenes" in globals() else pd.DataFrame())
if not outlier_animation_results.empty:
    outlier_display_cols = ["outlier_rank", "player_name", "ranking_event_type", "event_type", "event_id", "match_key", "game_time", "outlier_score", "output_path", "status"]
    display(outlier_animation_results[[col for col in outlier_display_cols if col in outlier_animation_results.columns]])
    print(f"Wrote {sum(outlier_animation_results['status'] == 'rendered')} outlier animation(s) under {OUTLIER_ANIMATION_DIR}")


def add_player_event_type_counts(ranked: pd.DataFrame) -> pd.DataFrame:
    work = ranked.reset_index(drop=True).copy()
    if "ranking_event_type" not in work.columns:
        work["ranking_event_type"] = work.apply(lambda row: row_event_count_label(row.to_dict()), axis=1)

    event_ids = work["event_id"] if "event_id" in work.columns else pd.Series(index=work.index, dtype=object)
    event_count_keys = [
        f"row_{idx}" if is_missing_value(event_id) else str(event_id)
        for idx, event_id in event_ids.items()
    ]
    work["_event_count_key"] = event_count_keys
    # Count unique event IDs when available, so duplicate role rows do not inflate the denominator.
    work["player_event_type_count"] = (
        work.groupby(["player_id", "ranking_event_type"])["_event_count_key"]
        .transform("nunique")
        .astype(int)
    )
    return work.drop(columns=["_event_count_key"])


def embedding_columns_for_diversity(df: pd.DataFrame) -> list[str]:
    pca_cols = [col for col in ["pca_1", "pca_2", "pca_3"] if col in df.columns]
    if pca_cols:
        return pca_cols
    return [col for col in ["embedding_x", "embedding_y"] if col in df.columns]


def scene_embedding_distance(row_a: dict, row_b: dict, embedding_cols: list[str]) -> float:
    if not embedding_cols:
        return np.inf
    a = np.array([numeric_value(row_a.get(col)) for col in embedding_cols], dtype="float64")
    b = np.array([numeric_value(row_b.get(col)) for col in embedding_cols], dtype="float64")
    valid = np.isfinite(a) & np.isfinite(b)
    if not valid.any():
        return np.inf
    return float(np.linalg.norm(a[valid] - b[valid]))


def player_embedding_distance_threshold(group: pd.DataFrame, embedding_cols: list[str]) -> float:
    if not embedding_cols or len(group) < 3:
        return 0.0
    coords = group[embedding_cols].apply(pd.to_numeric, errors="coerce").dropna().to_numpy(dtype="float64")
    if len(coords) < 3:
        return 0.0
    distances = []
    for i in range(len(coords)):
        for j in range(i + 1, len(coords)):
            distance = float(np.linalg.norm(coords[i] - coords[j]))
            if np.isfinite(distance) and distance > 0:
                distances.append(distance)
    if not distances:
        return 0.0
    return float(np.nanquantile(distances, DIVERSITY_MIN_PCA_DISTANCE_QUANTILE))


def event_key_for_diversity(row: dict) -> str | None:
    event_id = row.get("event_id")
    if is_missing_value(event_id):
        return None
    return str(event_id)


def has_same_event_id(candidate: dict, selected_rows: list[dict]) -> bool:
    candidate_key = event_key_for_diversity(candidate)
    return bool(candidate_key and any(event_key_for_diversity(row) == candidate_key for row in selected_rows))


def scene_diversity_rejection_reason(
    candidate: dict,
    selected_rows: list[dict],
    embedding_cols: list[str],
    min_embedding_distance: float,
) -> str | None:
    for selected in selected_rows:
        if has_same_event_id(candidate, [selected]):
            return "same_event"

        if display_value(candidate.get("match_key"), "") == display_value(selected.get("match_key"), ""):
            candidate_anchor = candidate.get("frame_anchor")
            selected_anchor = selected.get("frame_anchor")
            if not is_missing_value(candidate_anchor) and not is_missing_value(selected_anchor):
                fps = numeric_value(candidate.get("fps"), numeric_value(selected.get("fps"), 50.0))
                fps = 50.0 if not np.isfinite(fps) or fps <= 0 else fps
                seconds_apart = abs(float(candidate_anchor) - float(selected_anchor)) / max(fps, 1.0)
                if seconds_apart < DIVERSITY_MIN_SCENE_GAP_SECONDS:
                    return "near_time"

        if min_embedding_distance > 0:
            distance = scene_embedding_distance(candidate, selected, embedding_cols)
            if distance < min_embedding_distance:
                return "near_pca"
    return None


def select_diverse_player_group(group: pd.DataFrame, top_k: int) -> list[dict]:
    sorted_group = group.sort_values(["signature_score", "motion_energy"], ascending=[False, False])
    if not DIVERSIFY_PLAYER_TOP_SCENES:
        rows = sorted_group.head(top_k).to_dict("records")
        for row in rows:
            row["selection_note"] = "score_only"
        return rows

    embedding_cols = embedding_columns_for_diversity(sorted_group)
    min_embedding_distance = player_embedding_distance_threshold(sorted_group, embedding_cols)
    selected_rows = []
    rejected_rows = []
    for row in sorted_group.to_dict("records"):
        reason = scene_diversity_rejection_reason(row, selected_rows, embedding_cols, min_embedding_distance)
        if reason is None:
            row["selection_note"] = "highest_score" if not selected_rows else "diverse"
            selected_rows.append(row)
            if len(selected_rows) >= top_k:
                return selected_rows
        else:
            row["selection_note"] = f"skipped_{reason}"
            rejected_rows.append(row)

    selected_source_indices = {row.get("_source_index") for row in selected_rows}
    for row in sorted_group.to_dict("records"):
        if len(selected_rows) >= top_k:
            break
        if row.get("_source_index") in selected_source_indices or has_same_event_id(row, selected_rows):
            continue
        row["selection_note"] = "relaxed_diversity_fill"
        selected_rows.append(row)
        selected_source_indices.add(row.get("_source_index"))
    return selected_rows


def select_player_signature_scenes(ranked: pd.DataFrame, top_k_per_event: int = MP4S_PER_PLAYER_EVENT) -> pd.DataFrame:
    work = add_player_event_type_counts(ranked)
    work["global_rank"] = np.arange(1, len(work) + 1)
    work["_source_index"] = np.arange(len(work))
    selected_rows = []
    for _, group in work.groupby(["player_id", "ranking_event_type"], sort=False):
        selected_rows.extend(select_diverse_player_group(group, top_k=top_k_per_event))
    selected = pd.DataFrame(selected_rows)
    if selected.empty:
        return selected
    selected = selected.drop(columns=["_source_index"], errors="ignore")
    selected["player_event_scene_rank"] = selected.groupby(["player_id", "ranking_event_type"]).cumcount() + 1
    selected["player_scene_rank"] = selected["player_event_scene_rank"]
    return selected.sort_values(["player_name", "ranking_event_type", "player_event_scene_rank"]).reset_index(drop=True)


def render_player_signature_mp4s(ranked: pd.DataFrame) -> pd.DataFrame:
    selected = select_player_signature_scenes(ranked)
    if CLEAN_ANIMATION_DIR:
        for old_file in ANIMATION_DIR.glob("**/*.mp4"):
            old_file.unlink()
        old_dirs = sorted((path for path in ANIMATION_DIR.glob("**/*") if path.is_dir()), key=lambda path: len(path.parts), reverse=True)
        for old_dir in old_dirs:
            try:
                old_dir.rmdir()
            except OSError:
                pass
    results = []
    iterator = tqdm(list(selected.iterrows()), desc="Rendering player signature MP4s")
    for _, row in iterator:
        row_for_render = render_context_row(row.to_dict())
        output_file = animation_output_path(row_for_render)
        if output_file.exists() and not OVERWRITE_ANIMATIONS:
            results.append({**{col: row_for_render.get(col) for col in metadata_cols if col in row_for_render}, "output_path": str(output_file), "status": "exists"})
            continue
        try:
            rendered_path = render_scene_mp4(row_for_render, output_file)
            results.append({**{col: row_for_render.get(col) for col in metadata_cols if col in row_for_render}, "output_path": str(rendered_path), "status": "rendered"})
        except Exception as exc:
            results.append({**{col: row_for_render.get(col) for col in metadata_cols if col in row_for_render}, "output_path": str(output_file), "status": "failed", "error": str(exc)})
    return pd.DataFrame(results)


animation_results = render_player_signature_mp4s(ranked_scenes)
display(animation_results[["player_name", "ranking_event_type", "player_event_type_count", "player_event_scene_rank", "selection_note", "match_key", "game_time", "event_type", "action_type", "event_outcome", "signature_score", "frame_start", "frame_end", "output_path", "status"]])
print(f"Wrote {sum(animation_results['status'] == 'rendered')} player signature animation(s) under {ANIMATION_DIR}")


Rendering outlier MP4s:   0%|          | 0/2 [00:00<?, ?it/s]

,outlier_rank,player_name,ranking_event_type,event_type,event_id,match_key,game_time,outlier_score,output_path,status
0,1,Michael Olise,Carry,Carry,18909309801040,2025_11_08_FCU_FCB,061:43:96,653.496552,signature_scene_outputs/outliers/rank_01_micha...,rendered
1,2,K. Laimer,Carry,Carry,18909309800708,2025_11_08_FCU_FCB,046:52:84,264.933655,signature_scene_outputs/outliers/rank_02_k_lai...,rendered


Wrote 2 outlier animation(s) under signature_scene_outputs/outliers


Rendering player signature MP4s:   0%|          | 0/51 [00:00<?, ?it/s]

,player_name,ranking_event_type,player_event_type_count,player_event_scene_rank,selection_note,match_key,game_time,event_type,action_type,event_outcome,signature_score,frame_start,frame_end,output_path,status
0,A. Pavlović,Carry,29,1,highest_score,2025_10_04_SGE_FCB,073:43:72,Carry,Carry,None,0.743389,3612360,3612810,signature_scene_outputs/player_signature_anima...,rendered
1,A. Pavlović,Carry,29,2,diverse,2025_10_04_SGE_FCB,055:27:92,Carry,Carry,None,0.736778,3557570,3558020,signature_scene_outputs/player_signature_anima...,rendered
2,A. Pavlović,Carry,29,3,diverse,2025_09_13_FCB_HSV,054:22:59,Carry,Carry,None,0.714438,3564347,3564797,signature_scene_outputs/player_signature_anima...,rendered
3,Dayot Upamecano,Carry,70,1,highest_score,2025_10_04_SGE_FCB,004:10:03,Carry,Carry,None,0.831459,3343524,3343974,signature_scene_outputs/player_signature_anima...,rendered
4,Dayot Upamecano,Carry,70,2,diverse,2025_11_08_FCU_FCB,030:36:31,Carry,Carry,None,0.789362,2882199,2882649,signature_scene_outputs/player_signature_anima...,rendered
5,Dayot Upamecano,Carry,70,3,diverse,2025_11_08_FCU_FCB,010:36:48,Carry,Carry,None,0.744301,2822207,2822657,signature_scene_outputs/player_signature_anima...,rendered
6,Harry Kane,Carry,29,1,highest_score,2025_11_08_FCU_FCB,044:12:55,Carry,Carry,None,0.743693,2923011,2923461,signature_scene_outputs/player_signature_anima...,rendered
7,Harry Kane,Carry,29,2,diverse,2025_09_13_FCB_HSV,050:24:07,Carry,Carry,None,0.741109,3481947,3482397,signature_scene_outputs/player_signature_anima...,rendered
8,Harry Kane,Carry,29,3,diverse,2025_09_13_FCB_HSV,015:34:36,Carry,Carry,None,0.690198,3377461,3377911,signature_scene_outputs/player_signature_anima...,rendered
9,J. Stanišić,Carry,32,1,highest_score,2025_09_13_FCB_HSV,048:54:92,Carry,Carry,None,0.730927,3547963,3548413,signature_scene_outputs/player_signature_anima...,rendered


Wrote 51 player signature animation(s) under signature_scene_outputs/player_signature_animations


## Visual inspection

Use the embedding scatter to check whether selected scenes form player-specific islands or isolated outliers. The MP4 export above already renders the top scenes per player with metadata in the animation header; the cells below remain useful for quick interactive inspection inside the notebook.


In [11]:
PCA_HTML_PATH = OUTPUT_DIR / "interactive_pca_signature_scenes.html"
PCA_PREVIEW_DIR = OUTPUT_DIR / "interactive_pca_preview_animations"
PCA_PREVIEW_DIR.mkdir(parents=True, exist_ok=True)
OVERWRITE_PCA_PREVIEWS = False

pca_plot_df = ranked_scenes.copy().reset_index(drop=True)
pca_plot_df["_scene_row"] = np.arange(len(pca_plot_df))
if "ranking_event_type" not in pca_plot_df.columns:
    pca_plot_df["ranking_event_type"] = pca_plot_df.apply(lambda row: ranking_event_type_label(row.to_dict()), axis=1)

x_col = "pca_1" if "pca_1" in pca_plot_df.columns else "embedding_x"
y_col = "pca_2" if "pca_2" in pca_plot_df.columns else "embedding_y"
pc1_label = "PC1"
pc2_label = "PC2"
if hasattr(reducer, "explained_variance_ratio_") and len(reducer.explained_variance_ratio_) >= 2:
    pc1_label = f"PC1 ({100 * reducer.explained_variance_ratio_[0]:.1f}% variance)"
    pc2_label = f"PC2 ({100 * reducer.explained_variance_ratio_[1]:.1f}% variance)"

try:
    import ipywidgets as widgets
    from IPython.display import Video, clear_output, display
except Exception:
    widgets = None
    Video = None
    clear_output = None


def pca_preview_output_path(row: dict) -> Path:
    player = safe_filename(display_value(row.get("player_name"), "player"))
    event_label = safe_filename(row_event_count_label(row))
    event_id = display_value(row.get("event_id"), "")
    if event_id:
        scene_key = safe_filename(event_id)
    else:
        scene_key = safe_filename(f"{row.get('match_key')}_{row.get('frame_anchor')}")
    return PCA_PREVIEW_DIR / f"{player}_{event_label}_{scene_key}.mp4"


def selected_scene_summary(row: pd.Series) -> pd.DataFrame:
    cols = [
        "player_name", "ranking_event_type", "event_outcome", "signature_score", "match_key",
        "game_time", "frame_start", "frame_end", "event_id",
    ]
    available = [col for col in cols if col in row.index]
    return pd.DataFrame([{col: row[col] for col in available}])


def render_pca_selection(scene_row_index: int, preview_output, status_widget=None):
    scene_row_index = int(scene_row_index)
    row = ranked_scenes.iloc[scene_row_index]
    row_dict = row.to_dict()
    output_path = pca_preview_output_path(row_dict)
    if status_widget is not None:
        status_widget.value = (
            f"<b>Selected:</b> {display_value(row_dict.get('player_name'))} | "
            f"{display_value(row_dict.get('ranking_event_type'), display_value(row_dict.get('event_type')))} | "
            f"score {format_number(row_dict.get('signature_score'), 3)}"
        )
    with preview_output:
        clear_output(wait=True)
        display(selected_scene_summary(row))
        if OVERWRITE_PCA_PREVIEWS or not output_path.exists():
            print(f"Rendering preview to {output_path} ...")
            render_scene_mp4(row_dict, output_path)
        else:
            print(f"Using existing preview {output_path}")
        display(Video(str(output_path), embed=False, html_attributes="controls loop style='max-width: 900px; width: 100%;'"))


TABLEAU20 = [
    "#1f77b4", "#aec7e8", "#ff7f0e", "#ffbb78", "#2ca02c",
    "#98df8a", "#d62728", "#ff9896", "#9467bd", "#c5b0d5",
    "#8c564b", "#c49c94", "#e377c2", "#f7b6d2", "#7f7f7f",
    "#c7c7c7", "#bcbd22", "#dbdb8d", "#17becf", "#9edae5",
]


if px is not None:
    scatter_fig = px.scatter(
        pca_plot_df,
        x=x_col,
        y=y_col,
        color="player_name",
        color_discrete_sequence=TABLEAU20,
        symbol="ranking_event_type",
        hover_name="player_name",
        custom_data=["_scene_row"],
        hover_data={
            "ranking_event_type": True,
            "signature_score": ":.3f",
            "distinctiveness_ratio": ":.3f",
            "repeatability": ":.3f",
            "motion_energy": ":.3f",
            "same_player_nn_distance": ":.3f",
            "other_player_nn_distance": ":.3f",
            "match_key": True,
            "team_name": True,
            "jersey": True,
            "frame_start": True,
            "frame_end": True,
            x_col: ":.3f",
            y_col: ":.3f",
        },
        labels={x_col: pc1_label, y_col: pc2_label, "ranking_event_type": "event type"},
        title="Interactive PCA of candidate signature scenes",
        height=650,
        template="plotly_white",
    )
    scatter_fig.update_traces(marker={"size": 8, "opacity": 0.75})
    scatter_fig.update_layout(
        hovermode="closest",
        legend_title_text="Player",
        paper_bgcolor="white",
        plot_bgcolor="white",
        clickmode="event+select",
        dragmode="select",
    )
    scatter_fig.write_html(PCA_HTML_PATH, include_plotlyjs=True)
    print(f"Wrote interactive PCA to {PCA_HTML_PATH}")

    if widgets is not None and go is not None:
        try:
            fig_widget = go.FigureWidget(scatter_fig)
            preview_output = widgets.Output()
            status = widgets.HTML("Click a point, or box-select points, to render the first selected scene below.")

            def handle_points(trace, points, _state):
                if not points.point_inds:
                    return
                point_idx = points.point_inds[0]
                custom = trace.customdata[point_idx]
                scene_row_index = custom[0] if isinstance(custom, (list, tuple, np.ndarray)) else custom
                render_pca_selection(scene_row_index, preview_output, status_widget=status)

            for trace in fig_widget.data:
                trace.on_click(handle_points)
                trace.on_selection(handle_points)

            display(widgets.VBox([fig_widget, status, preview_output]))
        except Exception as exc:
            scatter_fig.show()
            print(f"Interactive click-to-render preview is unavailable in this kernel: {exc}")
    else:
        scatter_fig.show()
        print("Install/enable ipywidgets and plotly FigureWidget support for click-to-render previews inside the notebook.")
else:
    ax = pca_plot_df.plot.scatter(x=x_col, y=y_col, c="signature_score", colormap="viridis", figsize=(8, 6))
    ax.set_facecolor("white")
    ax.figure.set_facecolor("white")


Wrote interactive PCA to signature_scene_outputs/interactive_pca_signature_scenes.html


    'data': [{'customdata': array([[0, 'Carry', 0.91580547112462, ..., 4, 285961…

In [12]:
def pose_for_scene(scene_row, frame_position: str = "middle"):
    scene = extract_scene(scene_row.to_dict() if hasattr(scene_row, "to_dict") else dict(scene_row))
    if scene is None:
        raise ValueError("No skeleton data found for this scene")
    if frame_position == "start":
        idx = 0
    elif frame_position == "end":
        idx = -1
    else:
        idx = len(scene["poses"]) // 2
    pose = fill_nan_sequence(scene["poses"])[idx]
    frame_number = int(scene["frames"][idx])
    return pose, frame_number


def plot_pose_3d(scene_row, frame_position: str = "middle"):
    pose, frame_number = pose_for_scene(scene_row, frame_position=frame_position)
    title = f"{scene_row['player_name']} | {scene_row['event_type']} | frame {frame_number}"
    if go is None:
        import matplotlib.pyplot as plt
        fig = plt.figure(figsize=(7, 7))
        ax = fig.add_subplot(111, projection="3d")
        ax.scatter(pose[:, 0], pose[:, 1], pose[:, 2])
        for a, b in BONES:
            p1, p2 = pose[a - 1], pose[b - 1]
            ax.plot([p1[0], p2[0]], [p1[1], p2[1]], [p1[2], p2[2]], color="black")
        ax.set_title(title)
        return fig

    traces = []
    for a, b in BONES:
        p1, p2 = pose[a - 1], pose[b - 1]
        traces.append(go.Scatter3d(
            x=[p1[0], p2[0]], y=[p1[1], p2[1]], z=[p1[2], p2[2]],
            mode="lines", line={"width": 6, "color": "#1f77b4"}, showlegend=False,
        ))
    traces.append(go.Scatter3d(
        x=pose[:, 0], y=pose[:, 1], z=pose[:, 2],
        mode="markers+text",
        marker={"size": 4, "color": "#d62728"},
        text=[PART_NAMES[i] for i in range(1, 22)],
        textposition="top center",
        showlegend=False,
    ))
    fig = go.Figure(data=traces)
    fig.update_layout(
        title=title,
        scene={"aspectmode": "data", "xaxis_title": "x", "yaxis_title": "y", "zaxis_title": "z"},
        height=650,
    )
    fig.show()
    return fig


best_scene = ranked_scenes.iloc[0]
plot_pose_3d(best_scene)


## Optional: render a specific MP4 preview

The export cell already renders up to three MP4s per Bayern player and top-level event. Use this optional cell when you want to render a different row from `ranked_scenes`.


In [13]:
# Example: render any specific scene on demand.
# render_scene_mp4(ranked_scenes.iloc[0])

## Reading the output

The generated MP4s live under `signature_scene_outputs/player_signature_animations/`. Each player has one full-name folder with the number of unique top-level events of the selected type, such as `harry_kane_27_events/`, containing event-aware files such as `carry_rank_01.mp4`, `carry_rank_02.mp4`, and `carry_rank_03.mp4` when enough diverse scenes are available. Automatic feature-space outliers are removed before PCA/ranking and rendered separately under `signature_scene_outputs/outliers/`. The count is not split by subevent or outcome metadata. Within each player/event group, the export first takes the highest-scoring scene, then skips exact same-event repeats, strongly overlapping windows, and very nearby PCA neighbors before filling the next ranks. Each animation uses fixed player-centered scaling, draws the raw parquet joint coordinates without pose normalization or interpolation, and overlays the same-frame parquet ball when it is inside the fixed view. The orange marker and short trail fall back to `Positions_*.xml` only when parquet ball data is missing. Missing raw joints are left missing. The header stays concise with player, shirt, position, event, outcome when available, match, match time, clip duration, and score.

The highest-ranked scenes are not guaranteed to be `the truth`; they are a review queue. A coach, analyst, or editor should look at the per-player clips and mark which ones are genuinely meaningful. Those labels then become training data for a stronger supervised or foundation-model-based ranker.

Useful next steps:

- Replace the hand-crafted embedding with a self-supervised skeleton transformer encoder.
- Add ball proximity and opponent pressure features from positions data and surrounding players.
- Deduplicate adjacent windows so one long action does not dominate the top results.
- Train per-position baselines, because a goalkeeper's signature movements are not comparable to a winger's.
- Add human labels: `iconic`, `routine`, `bad-tracking`, `not-interesting`, and fine-tune the score weights.
